In [1]:
# ============================================================
# 039_iterative_notebook_builder_and_repair_loop
# ============================================================
#
# Overview
# ----------------
# Extends the Structure-Driven Notebook Generator (026) with an iterative,
# *cellwise* build-and-test-and-repair loop.
#
# After creating a skeleton notebook from Cell 00 (the structure source of truth),
# the generator fills cells one at a time, executes a deterministic prefix of the
# notebook (Cell 00..current), captures per-cell outputs, and automatically repairs
# failures using OpenAI-generated repair prompts sent back to Claude.
#
# The loop advances to the next cell ONLY when the execution prefix succeeds.
# This design minimizes "moving target" failures and makes each automated judgment
# inspectable via captured outputs.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Target notebook path (e.g., "target_notebook.ipynb")
#   - Task description / specification for the notebook to generate (SPEC_TEXT)
#   - Anthropic API key (for Claude authoring only)
#   - OpenAI API key (for repair prompt generation only)
#   - Max repair attempts per step (default: 3)
#   - Execution mode:
#       - "prefix": execute cells 0..i (recommended; stable + dependency-safe)
#       - "full":   execute the full notebook (optional; more brittle)
#   - Output inspection settings:
#       - always capture stdout/stderr + cell outputs for each execution
#       - optionally treat "soft failures" (bad output patterns) as failures
#
# Outputs:
#   - Fully generated and validated .ipynb file (saved in-place)
#   - Execution logs showing patch/test/repair cycles
#   - Per-step execution artifact:
#       - failing_cell index (if any)
#       - captured outputs per executed cell (stdout/stderr/result/error)
#   - Diagnostic report if any step fails after max attempts
#
# Structure
# ----------------
# Phase 0 (Common utilities)
# Cell 01: Imports & environment (nbformat, nbclient, anthropic, openai, json, pathlib, logging)
# Cell 02: Configure logging and API clients (Claude, OpenAI)
# Cell 03: Notebook I/O helpers:
#          - load_notebook(path) -> nbformat.NotebookNode
#          - save_notebook(nb, path)
#
# Phase 1 (Skeleton)
# Cell 04: Execute helper for skeleton generation:
#          - generate_cell00_via_claude(spec_text) -> str
#          - parse_structure_from_cell00(cell00_source) -> List[str]
#          - assemble_skeleton_from_structure(cell00_source, structure_lines) -> NotebookNode
#
# Phase 2 (Execution + outputs)
# Cell 05: Execution engine (prefix mode recommended):
#          - execute_prefix_with_outputs(path, upto_index) -> (ok: bool, info: dict)
#            info includes per-cell outputs and failing_cell (if any)
#          - render_outputs(info) for human inspection
#          - soft-failure detection (optional patterns)
#
# Phase 3 (Patch + validate)
# Cell 06: Patch helpers:
#          - build_patch_prompt_from_context(nb, target_indices, context_upto) -> str
#          - call_claude_patch(...) -> patch JSON (strict)
#          - strict validation: updates only requested indices; JSON closed; no extra text
#          - header preservation: updated cell MUST start with existing mandatory header block verbatim
#          - apply_patch_to_notebook(path, patch, preserve_header=True)
#
# Phase 4 (Repair loop)
# Cell 07: Repair prompt builder (OpenAI):
#          - build_repair_prompt_via_openai(cell00, cell_goal, failing_source, error_info) -> str
#            MUST instruct: copy existing header verbatim; minimal fix; do not move logic to other cells
#
# Cell 08: Iterative cellwise loop:
#          - for each planned cell i:
#              1) patch cell i (Claude)
#              2) execute prefix 0..i capturing outputs
#              3) if fail at cell j:
#                    - create repair prompt with OpenAI using:
#                      (Cell00 + goal(j) + current source(j) + error info + outputs summary)
#                    - patch failing cell j (Claude)
#                    - re-run prefix 0..i
#                 repeat up to max attempts
#              4) proceed only when prefix execution succeeds
#
# Phase 5 (Orchestrator + smoke tests)
# Cell 09: Smoke test runner:
#          - generate_notebook_with_repair_loop(spec_text, target_path)
#            -> returns final path + run report
#          - dry run (execute prefix + show outputs)
#          - phase2 run (patch+execute+repair) on a subset
#
# Notes
# ----------------
# - Cell 00 "# Structure" is the single source of truth for cell order and goals.
# - Every generated cell MUST preserve the mandatory header block exactly.
# - The generated target notebook must NOT include Anthropic/Claude runtime calls.
# - Claude output MUST be strict JSON patches; if JSON is malformed, retry in tight mode.
# - Execution uses a fresh kernel each run (nbclient), to avoid hidden in-memory state.
# - "Success" is defined as:
#     - no exception outputs in executed prefix, AND
#     - (optional) no soft-failure patterns detected in captured outputs.
# - Clear logging at each step: patched cell(s), execution range, failing cell, attempt count.
# - If max attempts exceeded, stop and emit a diagnostic report including:
#     - failing cell index, error details, and captured outputs for the executed prefix.
#
# Naming
# ----------------
# Suggested filename:
#   039_iterative_notebook_builder_and_repair_loop.ipynb
#
pass


In [2]:
# ============================================================
# Cell 1 — Imports & Environment
# ============================================================
# Overview:
#   Initializes the execution environment for the notebook generator.
#   Centralizes imports, explicitly loads env vars from env.txt,
#   and initializes API clients required for authoring-time LLM usage.
#
# Inputs / Outputs:
#   Inputs:
#     - env.txt:
#         Must contain ANTHROPIC_API_KEY.
#         May contain OPENAI_API_KEY (optional at this stage).
#   Outputs:
#     - claude_client: Anthropic client (authoring-time only)
#     - openai_client: OpenAI client (optional)
#     - SKILLS_JSONL_PATH: default runtime skill store path
#
# Notes:
#   - This generator notebook may use Claude at authoring time.
#   - Generated target notebooks must NOT depend on Claude at runtime.
#

# ------------------------------------------------------------
# Standard library imports
# ------------------------------------------------------------
import os
import json
import re
import datetime
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

# ------------------------------------------------------------
# Third-party imports
# ------------------------------------------------------------
from dotenv import load_dotenv

import nbformat
from nbformat.v4 import (
    new_notebook,
    new_code_cell,
)

# UI / display (kept here for convenience across cells)
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

# ------------------------------------------------------------
# Lightweight logging helpers
# ------------------------------------------------------------
def log_info(msg: str):
    print(f"[INFO] {msg}")

def log_warn(msg: str):
    print(f"[WARN] {msg}")

def log_error(msg: str):
    print(f"[ERROR] {msg}")

# ------------------------------------------------------------
# Environment loading (MANDATORY)
# ------------------------------------------------------------
ENV_PATH = Path("env.txt")
if not ENV_PATH.exists():
    raise FileNotFoundError("env.txt not found. Please create env.txt in the working directory.")

load_dotenv(str(ENV_PATH))

# ------------------------------------------------------------
# Skill store configuration (for Skill-aware generation)
# ------------------------------------------------------------
# #027 default output. You can override via env.txt: SKILLS_JSONL_PATH=...
SKILLS_JSONL_PATH = Path(os.getenv("SKILLS_JSONL_PATH", "skills/human_corrections.jsonl"))

# ------------------------------------------------------------
# Claude (Anthropic) configuration — AUTHORING TIME ONLY
# ------------------------------------------------------------
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY could not be loaded from env.txt. "
        "Claude is required for structure and cell authoring."
    )
log_info("✅ ANTHROPIC_API_KEY loaded successfully")

from anthropic import Anthropic
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)

# ------------------------------------------------------------
# OpenAI configuration — OPTIONAL (runtime notebooks only)
# ------------------------------------------------------------
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
openai_client = None

if OPENAI_API_KEY:
    from openai import OpenAI
    openai_client = OpenAI(api_key=OPENAI_API_KEY)
    log_info("ℹ️ OPENAI_API_KEY loaded (OpenAI client available if needed)")
else:
    log_info("ℹ️ OPENAI_API_KEY not found. This is acceptable for Phase 1 (structure generation).")

# ------------------------------------------------------------
# Runtime metadata (for logging / provenance)
# ------------------------------------------------------------
# Use timezone-aware UTC timestamps to align with Notion/date fields and cross-system logs.
GENERATOR_STARTED_AT_UTC = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
log_info(f"Generator started at (UTC): {GENERATOR_STARTED_AT_UTC}")

# Optional: show skill store path (does not leak secrets)
log_info(f"Skill store JSONL path: {SKILLS_JSONL_PATH}")


[INFO] ✅ ANTHROPIC_API_KEY loaded successfully
[INFO] ℹ️ OPENAI_API_KEY loaded (OpenAI client available if needed)
[INFO] Generator started at (UTC): 2026-02-06T04:10:35+00:00
[INFO] Skill store JSONL path: skills/human_corrections.jsonl


In [3]:
# ============================================================
# Cell 1b (NEW) — Skill Store Loader & Retriever
# ============================================================
# Overview:
#   Loads your local Skill Store (JSONL produced by #027 pipeline) and provides
#   lightweight retrieval utilities to inject the most relevant "human correction"
#   skills into prompts (e.g., for Cell 2 Cell00 generation, or Phase-2 patching).
#
# Inputs / Outputs:
#   Inputs:
#     - skills/human_corrections.jsonl (default)  OR any JSONL path you specify
#   Outputs:
#     - SKILL_STORE (in-memory list of dict records)
#     - Helper functions:
#         - load_skill_store(jsonl_path)
#         - retrieve_skills(query, top_k=8, ...)
#         - format_skills_for_prompt(skills, max_chars=6000)
#         - build_skill_augmented_spec(spec_text, query, top_k=8, ...)
#
# Notes:
#   - No external dependencies (no embeddings). Uses a simple lexical scoring.
#   - Works even if JSONL schema evolves: ignores unknown fields safely.
#   - Retrieval is "good enough" for MVP; later you can swap in embeddings.
#

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional

# -----------------------------
# Safe fallbacks
# -----------------------------
try:
    log_info
except NameError:
    def log_info(msg: str): print(f"[INFO] {msg}")
    def log_warn(msg: str): print(f"[WARN] {msg}")
    def log_error(msg: str): print(f"[ERROR] {msg}")

DEFAULT_SKILL_JSONL_PATH = Path("skills/human_corrections.jsonl")

# In-memory store
SKILL_STORE: List[Dict[str, Any]] = []
SKILL_STORE_PATH: Path = DEFAULT_SKILL_JSONL_PATH


# -----------------------------
# Utilities
# -----------------------------
_STOPWORDS = {
    "the","a","an","and","or","to","of","in","on","for","with","as","by","is","are","was","were",
    "be","been","being","that","this","these","those","it","its","from","at","into","over","under",
    "we","you","they","i","he","she","them","our","your","their","not","do","does","did","done",
    "can","could","should","would","may","might","will","just","only","also","very","more","most",
    "etc","via","per"
}

def _normalize_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.lower()
    s = re.sub(r"[\u3000\s]+", " ", s)  # normalize whitespace (incl JP full-width)
    return s.strip()

def _tokenize(s: str) -> List[str]:
    s = _normalize_text(s)
    # simple word-like tokenization (keeps numbers and underscores)
    toks = re.findall(r"[a-z0-9_]+", s)
    toks = [t for t in toks if t and t not in _STOPWORDS and len(t) >= 2]
    return toks

def _safe_get(d: Dict[str, Any], key: str, default="") -> str:
    v = d.get(key, default)
    return "" if v is None else str(v)

def _build_record_text(rec: Dict[str, Any]) -> str:
    # Make a single searchable text blob from common fields
    fields = [
        _safe_get(rec, "title"),
        _safe_get(rec, "correction_type"),
        _safe_get(rec, "old_approach"),
        _safe_get(rec, "new_approach"),
        _safe_get(rec, "reasoning"),
        _safe_get(rec, "context_snippet"),
        _safe_get(rec, "source_file"),
        _safe_get(rec, "skill_id"),
        _safe_get(rec, "tags"),
    ]
    return _normalize_text("\n".join([f for f in fields if f]))


# -----------------------------
# Loader
# -----------------------------
def load_skill_store(jsonl_path: Path = DEFAULT_SKILL_JSONL_PATH) -> List[Dict[str, Any]]:
    """
    Load Skill Store from a JSONL file.

    Expected each line: a JSON object containing at least:
      - id (recommended)
      - title, correction_type, old_approach, new_approach, reasoning (recommended)
    """
    global SKILL_STORE, SKILL_STORE_PATH
    SKILL_STORE_PATH = Path(jsonl_path)

    if not SKILL_STORE_PATH.exists():
        log_warn(f"Skill store not found: {SKILL_STORE_PATH}")
        SKILL_STORE = []
        return SKILL_STORE

    records: List[Dict[str, Any]] = []
    bad = 0

    with SKILL_STORE_PATH.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    records.append(obj)
                else:
                    bad += 1
            except Exception:
                bad += 1

    SKILL_STORE = records
    log_info(f"Loaded {len(SKILL_STORE)} skill record(s) from {SKILL_STORE_PATH} (bad lines: {bad})")
    return SKILL_STORE


# -----------------------------
# Retriever (simple lexical scoring)
# -----------------------------
def _score_record(query_tokens: List[str], rec: Dict[str, Any]) -> float:
    """
    Lexical scoring:
      - token overlap count (weighted)
      - small boosts for title matches and correction_type matches
    """
    blob = _build_record_text(rec)
    blob_tokens = set(_tokenize(blob))

    overlap = 0
    for t in query_tokens:
        if t in blob_tokens:
            overlap += 1

    if overlap == 0:
        return 0.0

    # boost title hits
    title = _normalize_text(_safe_get(rec, "title"))
    title_tokens = set(_tokenize(title))
    title_overlap = sum(1 for t in query_tokens if t in title_tokens)

    # boost correction_type (if query mentions it)
    ctype = _normalize_text(_safe_get(rec, "correction_type"))
    ctype_boost = 0
    if ctype and any(t in ctype for t in query_tokens):
        ctype_boost = 1

    # final score
    return overlap + (0.7 * title_overlap) + (0.3 * ctype_boost)


def retrieve_skills(
    query: str,
    top_k: int = 8,
    min_score: float = 1.0,
    prefer_types: Optional[List[str]] = None,
    store: Optional[List[Dict[str, Any]]] = None
) -> List[Dict[str, Any]]:
    """
    Retrieve top-k skill records by lexical match.

    Args:
      query: text describing what you want (e.g. "notebook generator structure parsing")
      top_k: number of records to return
      min_score: filter out weak matches
      prefer_types: optional list like ["Method","Structure"] to slightly boost those types
      store: optionally pass a list of records; otherwise uses SKILL_STORE

    Returns:
      List of records (dicts), sorted best->worst
    """
    store = store if store is not None else SKILL_STORE
    if not store:
        log_warn("Skill store is empty. Call load_skill_store() first.")
        return []

    q_tokens = _tokenize(query)
    if not q_tokens:
        return []

    scored: List[Tuple[float, Dict[str, Any]]] = []
    prefer_types_norm = [t.strip().lower() for t in (prefer_types or []) if t]

    for rec in store:
        s = _score_record(q_tokens, rec)

        if prefer_types_norm and s > 0:
            ctype = _normalize_text(_safe_get(rec, "correction_type"))
            if any(pt == ctype for pt in prefer_types_norm):
                s += 0.5  # mild preference

        if s >= min_score:
            scored.append((s, rec))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [r for _, r in scored[:top_k]]


# -----------------------------
# Prompt formatting
# -----------------------------
def format_skills_for_prompt(
    skills: List[Dict[str, Any]],
    max_chars: int = 6000
) -> str:
    """
    Render a compact block for inclusion in an LLM prompt.
    Keeps it short and consistent, prioritizing actionable deltas.
    """
    if not skills:
        return ""

    lines: List[str] = []
    lines.append("SKILL STORE (high-value human corrections to follow):")
    lines.append("Use these as constraints / best practices when generating the notebook.")
    lines.append("")

    for i, s in enumerate(skills, 1):
        sid = _safe_get(s, "skill_id") or _safe_get(s, "id")
        title = _safe_get(s, "title")
        ctype = _safe_get(s, "correction_type")
        old_a = _safe_get(s, "old_approach")
        new_a = _safe_get(s, "new_approach")
        reason = _safe_get(s, "reasoning")

        block = [
            f"[Skill {i}]",
            f"- id: {sid}",
            f"- type: {ctype}",
            f"- title: {title}",
            f"- old_approach: {old_a}",
            f"- new_approach: {new_a}",
            f"- reasoning: {reason}",
            ""
        ]
        lines.extend(block)

        # enforce max_chars
        if len("\n".join(lines)) > max_chars:
            lines.append("...[TRUNCATED: skill context exceeded max_chars]...")
            break

    return "\n".join(lines).strip()


def build_skill_augmented_spec(
    spec_text: str,
    query: str,
    top_k: int = 8,
    prefer_types: Optional[List[str]] = None,
    max_skill_chars: int = 6000
) -> str:
    """
    Combine SPEC_TEXT with retrieved skill context.

    Typical usage:
      augmented = build_skill_augmented_spec(SPEC_TEXT, query="notebook generator", top_k=8)
      call_claude_cell00(augmented)
    """
    spec_text = (spec_text or "").strip()
    skills = retrieve_skills(query=query, top_k=top_k, prefer_types=prefer_types)

    skill_block = format_skills_for_prompt(skills, max_chars=max_skill_chars)
    if not skill_block:
        return spec_text

    return f"""{spec_text}

---

{skill_block}
""".strip()


# -----------------------------
# Quick start
# -----------------------------
# Load once (safe if file doesn't exist)
_ = load_skill_store(DEFAULT_SKILL_JSONL_PATH)

log_info("Skill Store Loader & Retriever ready ✅")
log_info(f"Skill store path: {SKILL_STORE_PATH}")
log_info("Example:")
log_info("  skills = retrieve_skills('notebook structure parsing', top_k=5, prefer_types=['Method','Structure'])")
log_info("  augmented_spec = build_skill_augmented_spec(SPEC_TEXT, query='notebook generator', top_k=8)")


[INFO] Loaded 40 skill record(s) from skills/human_corrections.jsonl (bad lines: 0)
[INFO] Skill Store Loader & Retriever ready ✅
[INFO] Skill store path: skills/human_corrections.jsonl
[INFO] Example:
[INFO]   skills = retrieve_skills('notebook structure parsing', top_k=5, prefer_types=['Method','Structure'])
[INFO]   augmented_spec = build_skill_augmented_spec(SPEC_TEXT, query='notebook generator', top_k=8)


In [4]:
# ============================================================
# Cell 2 — Claude Cell 00 Writer
# ============================================================
# Overview:
#   Asks Claude to generate ONLY Cell 00 for the target notebook as strict JSON.
#   Cell 00 is the single source of truth for notebook structure under "# Structure".
#
# Notes:
#   - Claude is used ONLY at authoring time (this generator notebook), never at runtime.
#   - Output MUST be strict JSON parseable by json.loads().
#

# ------------------------------------------------------------
# Imports (keep local to avoid execution-order issues)
# ------------------------------------------------------------
import os
import json
import time
from typing import Any, Dict

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------
try:
    claude_client
except NameError:
    raise NameError("claude_client is not initialized. Run Cell 1 first.")

try:
    log_info
except NameError:
    def log_info(msg: str): print(f"[INFO] {msg}")
    def log_warn(msg: str): print(f"[WARN] {msg}")
    def log_error(msg: str): print(f"[ERROR] {msg}")

# ------------------------------------------------------------
# Configuration (allow env override)
# ------------------------------------------------------------
CLAUDE_CELL00_MODEL = os.getenv("CLAUDE_CELL00_MODEL", "claude-sonnet-4-5-20250929")
CLAUDE_CELL00_MAX_TOKENS = int(os.getenv("CLAUDE_CELL00_MAX_TOKENS", "2800"))
CLAUDE_CELL00_MAX_RETRIES = int(os.getenv("CLAUDE_CELL00_MAX_RETRIES", "3"))

LAST_CLAUDE_CELL00: Dict[str, Any] = {}

# ------------------------------------------------------------
# Strict JSON schema for Claude output
# ------------------------------------------------------------
CELL00_JSON_SCHEMA = """
Return ONLY valid JSON. No prose. No markdown fences. No code fences. No backticks.
The JSON must be directly parseable by Python json.loads().

Schema:
{
  "notebook_id": "string",
  "cell00_source": "string"
}

Rules for notebook_id:
- Must be the notebook identifier used in the Cell 00 title line.
- Example: "023_daily_scanner_and_incremental_ingest"

Rules for cell00_source:
- MUST be Python CODE cell content (comment-style header).
- MUST start EXACTLY with:
  # ============================================================
  # <NOTEBOOK_ID>
  # ============================================================
  #
  # Overview
  # ----------------
  #
  # Inputs / Outputs
  # ----------------
  #
  # Structure
  # ----------------
  #
  # Notes
  # ----------------
- Under "# Structure", you MUST include an explicit ordered list of cells.
- Each line MUST be formatted EXACTLY as:
    # Cell 01: <Short descriptive title>
    # Cell 02: <Short descriptive title>
- Do NOT use bullets ("-", "*") or plain text.
- Every structure line MUST start with "# Cell XX:".
- Use 10 to 16 cells total (Cell 01..Cell NN).
- Do NOT include any other "Cell XX:" lines outside the Structure section.
- Keep titles short and descriptive (no long sentences).
- Include brief content under Overview / Inputs / Outputs / Notes as comments, but do not overfill.
""".strip()

CELL00_SYSTEM_PROMPT = (
    "You are a notebook architect.\n"
    "Your task is to produce ONLY Cell 00 as strict JSON.\n"
    "Follow the schema and formatting requirements exactly.\n"
)

def _extract_first_json_object(text: str) -> str:
    """
    Extract the first top-level JSON object from a string.
    Robust to accidental leading/trailing text.
    """
    text = (text or "").strip()
    if not text:
        raise ValueError("Empty model output (expected JSON object).")

    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found in model output (missing '{').")

    depth = 0
    in_str = False
    esc = False
    end = None

    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is None:
        raise ValueError("JSON object appears incomplete (unmatched braces).")

    return text[start:end].strip()

def build_cell00_user_prompt(spec_text: str) -> str:
    """
    Build the Claude user prompt that requests ONLY Cell 00 JSON.
    """
    spec_text = (spec_text or "").strip()
    return f"""
Create ONLY Cell 00 for the requested notebook. Do NOT create any other cells.

{CELL00_JSON_SCHEMA}

SPEC_TEXT (verbatim):
<<<
{spec_text}
>>>
""".strip()

def _join_anthropic_text_blocks(resp) -> str:
    """
    Anthropic SDK response content may be a list of blocks.
    We join text blocks safely.
    """
    parts = []
    for block in getattr(resp, "content", []) or []:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
        else:
            # fallback
            parts.append(str(block))
    return "\n".join(parts).strip()

def call_claude_cell00(spec_text: str) -> Dict[str, Any]:
    """
    Call Claude and return:
      {"notebook_id": str, "cell00_source": str}
    """
    user_prompt = build_cell00_user_prompt(spec_text)

    last_err = None
    for attempt in range(1, CLAUDE_CELL00_MAX_RETRIES + 1):
        try:
            log_info(f"Calling Claude for Cell 00... ({attempt}/{CLAUDE_CELL00_MAX_RETRIES})")

            resp = claude_client.messages.create(
                model=CLAUDE_CELL00_MODEL,
                system=CELL00_SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_prompt}],
                max_tokens=CLAUDE_CELL00_MAX_TOKENS,
            )

            raw = _join_anthropic_text_blocks(resp)
            json_str = _extract_first_json_object(raw)
            obj = json.loads(json_str)

            # Minimal validation
            if not isinstance(obj, dict):
                raise ValueError("Cell00 response must be a JSON object.")
            if "notebook_id" not in obj or "cell00_source" not in obj:
                raise ValueError("Cell00 JSON must include 'notebook_id' and 'cell00_source'.")
            if not isinstance(obj["notebook_id"], str) or not obj["notebook_id"].strip():
                raise ValueError("'notebook_id' must be a non-empty string.")
            if not isinstance(obj["cell00_source"], str) or not obj["cell00_source"].strip():
                raise ValueError("'cell00_source' must be a non-empty string.")

            global LAST_CLAUDE_CELL00
            LAST_CLAUDE_CELL00 = obj
            log_info("✅ Cell 00 JSON received and validated.")
            return obj

        except Exception as e:
            last_err = e
            log_error(f"Cell 00 generation failed: {type(e).__name__}: {e}")
            if attempt < CLAUDE_CELL00_MAX_RETRIES:
                wait = 2 ** attempt
                log_info(f"Retrying in {wait}s...")
                time.sleep(wait)

    raise RuntimeError(f"Failed to generate Cell 00 after {CLAUDE_CELL00_MAX_RETRIES} attempts: {last_err}")


In [5]:
# ============================================================
# Cell 3 — Structure Parser & Skeleton Assembler (nbformat)
# ============================================================
# Overview:
#   Parses the "# Structure" section from Claude-generated Cell 00 and uses it as the
#   single source of truth to create a deterministic skeleton notebook:
#   - Cell 00: Claude's cell00_source (CODE cell)
#   - Cell 01..N: generated empty CODE cells with mandatory per-cell headers
#
# Conventions enforced:
#   - Cell 00 should start with the required header template (fail-fast check).
#   - EVERY generated cell starts with the mandatory per-cell header template.
#   - Environment loading (load_dotenv("env.txt")) is inserted into Cell 01 by default.
#
# Notes:
#   - Structure parsing is strict by design. If formatting deviates, fail fast and fix Cell 00.
#   - Canonical structure line format:
#       # Cell 01: <Title>
#       # Cell 02: <Title>
#

import re  # IMPORTANT: used in parsing
import nbformat
from nbformat.v4 import new_notebook, new_code_cell
from pathlib import Path
from typing import List, Tuple

LAST_STRUCTURE_ITEMS = None

# ------------------------------------------------------------
# Mandatory snippets inserted into generated skeleton cells
# ------------------------------------------------------------
ENV_LOAD_SNIPPET = (
    "# --- Mandatory env loading ---\n"
    "from dotenv import load_dotenv\n"
    "load_dotenv('env.txt')\n\n"
)

OPENAI_RUNTIME_SNIPPET = (
    "# --- Runtime LLM configuration (given / assumed) ---\n"
    "llm_provider = 'OpenAI'\n"
    "llm_model = 'gpt-4o-mini'\n"
    "llm_temperature = 0.0\n\n"
)

def per_cell_header(index: int, title: str) -> str:
    """
    EVERY cell must start with this header.
    """
    return (
        "# ============================================================\n"
        f"# Cell {index:02d} — {title}\n"
        "# ============================================================\n"
        "# Overview:\n"
        "#\n"
        "# Inputs / Outputs:\n"
        "#\n"
        "# Notes:\n"
        "#\n\n"
    )

def _assert_cell00_header(cell00_source: str) -> None:
    """
    Fail fast if Cell 00 does not start with the required notebook header template.
    Keeps errors close to the root cause (Cell 2 prompt).
    """
    src = (cell00_source or "").lstrip()
    required_prefix = (
        "# ============================================================\n"
        "# "
    )
    if not src.startswith(required_prefix):
        head = src[:300].replace("\n", "\\n")
        raise ValueError(
            "Cell 00 does not appear to start with the required header template.\n"
            "Expected it to start with:\n"
            "  # ============================================================\n"
            "  # <NOTEBOOK_ID>\n"
            "  # ============================================================\n"
            f"Got head:\n  {head}"
        )

def _find_structure_block_lines(cell00_source: str) -> List[str]:
    """
    Return the lines that belong to the Structure section, excluding the header lines.
    Stops when the Notes section begins.
    """
    lines = (cell00_source or "").splitlines()

    # Find "# Structure" and "# Notes"
    start_idx = None
    end_idx = None

    for i, line in enumerate(lines):
        if line.strip() == "# Structure":
            start_idx = i
            continue
        if start_idx is not None and line.strip() == "# Notes":
            end_idx = i
            break

    if start_idx is None:
        raise ValueError(
            "Cell 00 is missing the '# Structure' line.\n"
            "Ensure Cell 00 contains a section header exactly:\n"
            "  # Structure"
        )

    if end_idx is None:
        raise ValueError(
            "Cell 00 is missing the '# Notes' line after the Structure section.\n"
            "Ensure Cell 00 contains a section header exactly:\n"
            "  # Notes\n"
            "and it appears AFTER '# Structure'."
        )

    return lines[start_idx + 1 : end_idx]

def parse_structure_items(cell00_source: str) -> List[Tuple[int, str]]:
    """
    Canonical structure line format (required):
      # Cell 01: Title
      # Cell 02: Title

    Allowed noise lines inside the Structure block (ignored):
      # ----------------
      #
      # (empty comment line)

    Returns:
      [(1, "Title"), (2, "Title"), ...]
    """
    _assert_cell00_header(cell00_source)
    block_lines = _find_structure_block_lines(cell00_source)

    items: List[Tuple[int, str]] = []
    seen = set()

    for raw in block_lines:
        line = raw.strip()
        if not line:
            continue

        # Ignore separator lines like "# ----------------"
        if re.fullmatch(r"#\s*-{3,}", line):
            continue

        # Ignore empty-comment lines like "#" or "#   "
        if re.fullmatch(r"#\s*", line):
            continue

        # STRICT canonical format: "# Cell 01: Title"
        m = re.fullmatch(r"#\s*Cell\s+(\d{2})\s*:\s*(.+)", line)
        if not m:
            raise ValueError(
                "Invalid Structure line format.\n"
                f"Got: {raw}\n"
                "Expected exactly:\n"
                "  # Cell 01: <Title>\n"
                "Allowed noise lines:\n"
                "  #\n"
                "  # ----------------"
            )

        idx = int(m.group(1))
        title = m.group(2).strip()

        if not title:
            raise ValueError(f"Empty title in Structure line: {raw}")

        if idx in seen:
            raise ValueError(f"Duplicate cell index detected under # Structure: {idx:02d}")
        seen.add(idx)

        items.append((idx, title))

    if not items:
        raise ValueError("No '# Cell XX: Title' lines found under # Structure.")

    # Validate contiguous indices starting at 1
    indices = [i for i, _ in items]
    expected = list(range(1, len(indices) + 1))
    if indices != expected:
        raise ValueError(
            "Cell indices under # Structure must be contiguous starting from 01.\n"
            f"Got: {indices}\nExpected: {expected}\n"
            "Fix Cell 00 so it lists:\n"
            "  # Cell 01: ...\n"
            "  # Cell 02: ...\n"
            "  ... (no gaps, no reorder)"
        )

    global LAST_STRUCTURE_ITEMS
    LAST_STRUCTURE_ITEMS = items
    return items

def build_skeleton_notebook(cell00_source: str, items: List[Tuple[int, str]]) -> nbformat.NotebookNode:
    """
    Create a deterministic skeleton notebook:
      - Cell 00: Claude source
      - Cells 01..N: generated code cells with mandatory per-cell header
      - Insert env loading into Cell 01 by default
    """
    _assert_cell00_header(cell00_source)

    cells = [new_code_cell(cell00_source)]

    for idx, title in items:
        src = per_cell_header(idx, title)

        if idx == 1:
            src += ENV_LOAD_SNIPPET + OPENAI_RUNTIME_SNIPPET
            src += "# TODO: add shared imports and configuration used across cells\n"
        else:
            src += "# TODO: implement this cell\n"

        cells.append(new_code_cell(src))

    # Optional: store structure as metadata for traceability
    meta = {
        "generated_by": "026_Claude_Structure_Driven_Notebook_Generator",
        "structure_items": [{"cell": i, "title": t} for i, t in items],
    }
    return new_notebook(cells=cells, metadata=meta)

def save_skeleton_notebook(nb: nbformat.NotebookNode, outfile_path: str) -> Path:
    """
    Save skeleton notebook to disk (overwrite-safe).
    """
    out = Path(outfile_path).expanduser().resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nb, str(out))
    return out

def preview_saved_notebook(path: Path, n: int = 12) -> str:
    """
    Read the saved .ipynb and return a markdown preview of the first N cells.
    """
    nb = nbformat.read(str(path), as_version=4)
    lines = []
    lines.append(f"**Saved file:** `{path}`")
    lines.append(f"**Cells:** {len(nb.cells)}\n")

    for i, c in enumerate(nb.cells[:n]):
        src = (c.source or "").strip()
        head = src[:700] + ("..." if len(src) > 700 else "")
        lines.append(f"### Cell {i:02d} ({c.cell_type})")
        lines.append(f"```python\n{head}\n```")

    return "\n".join(lines)


In [6]:
# ============================================================
# Cell 4 — UI (ipywidgets)
# ============================================================
# Overview:
#   UI for the structure-driven notebook generator.
#   Users paste SPEC_TEXT, set outfile, optionally preview, and generate skeleton in Cell 5.
#   (Optional) Skill-related controls are included as placeholders for Cell 1b integration.
#
# Outputs (globals for later cells):
#   - LAST_SPEC_TEXT (str)
#   - LAST_SAVED_NOTEBOOK_PATH (Path)
#
# Notes:
#   - Button handler lives in Cell 5.
#   - This cell is safe to re-run (clears previous UI output and rebinds widgets).
#

from pathlib import Path

# ------------------------------------------------------------
# Shared state (globals)
# ------------------------------------------------------------
LAST_SPEC_TEXT = None
LAST_SAVED_NOTEBOOK_PATH = None

# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------
spec_text_w = widgets.Textarea(
    value="",
    description="SPEC_TEXT",
    placeholder="Paste the full notebook specification here...",
    layout=widgets.Layout(width="100%", height="420px")
)

outfile_w = widgets.Text(
    value="generated_notebook.ipynb",
    description="Outfile",
    layout=widgets.Layout(width="70%")
)

preview_w = widgets.Checkbox(
    value=True,
    description="Preview generated notebook"
)

preview_n_w = widgets.IntSlider(
    value=12,
    min=3,
    max=30,
    step=1,
    description="Preview cells",
    continuous_update=False,
    layout=widgets.Layout(width="60%")
)

# ---- Optional: Skill integration placeholders (used once Cell 1b is added) ----
use_skill_w = widgets.Checkbox(
    value=True,
    description="Use Skill Store (if available)",
    tooltip="If enabled and a Skill Store loader exists, the generator can retrieve relevant skills to refine prompts."
)

skill_query_w = widgets.Text(
    value="",
    description="Skill query",
    placeholder="Optional: keywords to retrieve relevant skills (e.g., notion, jsonl, retries)...",
    layout=widgets.Layout(width="70%")
)

skill_topk_w = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description="Top-K",
    continuous_update=False,
    layout=widgets.Layout(width="60%")
)

generate_btn = widgets.Button(
    description="Generate Skeleton (Cell00 → Structure → Cells)",
    button_style="primary",
    tooltip="Claude writes Cell 00, Python parses # Structure, then builds the skeleton notebook."
)

ui_out = widgets.Output()

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>Claude Structure-Driven Notebook Generator</h3>"),
    widgets.HTML(
        "<p><b>Phase 1:</b> Paste SPEC_TEXT → Claude writes Cell 00 (with <code># Structure</code>) → "
        "Python parses Structure → saves skeleton .ipynb.</p>"
        "<p><b>Phase 2:</b> Fill individual cells via Claude patches (Cell 6a/6b).</p>"
        "<p style='margin-top:8px;'><b>Skill Store:</b> Optional. If you added Cell 1b, the generator can pull "
        "high-value prior corrections to improve the prompt and reduce repeated mistakes.</p>"
    ),
    spec_text_w,
    widgets.HBox([outfile_w, preview_w]),
    preview_n_w,
    widgets.HBox([use_skill_w, skill_query_w]),
    skill_topk_w,
    generate_btn,
    ui_out,
])

# Clear and display UI (re-run safe)
clear_output(wait=True)
display(ui)
print("UI ready. Paste SPEC_TEXT and click 'Generate Skeleton (Cell00 → Structure → Cells)'.")


UI ready. Paste SPEC_TEXT and click 'Generate Skeleton (Cell00 → Structure → Cells)'.


In [14]:
# ============================================================
# Cell 5 — Phase 1 Orchestrator (Generate Skeleton)
# ============================================================
# Overview:
# This cell wires the UI button to the Phase 1 pipeline:
#   1) Read SPEC_TEXT from the widget
#   2) (Optional) Retrieve relevant Skills and inject into SPEC_TEXT
#   3) Call Claude to generate ONLY Cell 00 (strict JSON)
#   4) Parse the "# Structure" section from Cell 00
#   5) Deterministically assemble a skeleton notebook (Cell 00 + Cell 01..N)
#   6) Save the .ipynb to disk (overwrite-safe)
#   7) Persist results into globals for downstream reuse
#
# Inputs / Outputs:
# Inputs:
# - spec_text_w.value  (SPEC_TEXT)
# - outfile_w.value    (output path)
# - preview_w.value    (preview toggle)
# - preview_n_w.value  (how many cells to preview)
# - (optional) use_skill_w / skill_query_w / skill_topk_w  (from Cell 4)
# - (optional) retrieve_skills(query, top_k)               (from Cell 1b)
#
# Outputs (globals):
# - LAST_SPEC_TEXT (str)
# - LAST_CLAUDE_CELL00 (dict)          [set in Cell 2]
# - LAST_STRUCTURE_ITEMS (List[Tuple]) [set in Cell 3]
# - LAST_SAVED_NOTEBOOK_PATH (Path)
#
# Notes:
# - Handler is de-duplicated if the cell is re-run.
# - Fail-fast behavior:
#     - If Cell 00 JSON is invalid -> stop and show error
#     - If Structure parsing fails -> stop and show the offending lines
# - Skill integration is optional and will gracefully fallback if Cell 1b is not present.
#

import traceback
from typing import Optional

# ------------------------------------------------------------
# Safety: prevent duplicate handler attachment on re-run
# ------------------------------------------------------------
try:
    generate_btn._click_handlers.callbacks.clear()
except Exception:
    pass

_is_running = False


def _show_structure_items(items: List[Tuple[int, str]]) -> None:
    lines = ["## Parsed Structure Items"]
    for idx, title in items:
        lines.append(f"- Cell {idx:02d}: {title}")
    display(Markdown("\n".join(lines)))


def _format_skills_as_context(skills: List[Dict[str, Any]], max_items: int = 8) -> str:
    """
    Convert retrieved skill records into a compact context block for Claude.
    Assumes each skill has fields like:
      - title, correction_type, old_approach, new_approach, reasoning, source_file
    This function is intentionally defensive: missing keys are tolerated.
    """
    if not skills:
        return ""

    lines = []
    lines.append("## Skill Context (retrieved from Human Corrections DB)")
    lines.append("Use these as guardrails/lessons learned when designing the notebook.")
    lines.append("Do not copy verbatim; apply the intent.\n")

    for i, s in enumerate(skills[:max_items], 1):
        title = (s.get("title") or "untitled").strip()
        ctype = (s.get("correction_type") or s.get("type") or "").strip()
        old_a = (s.get("old_approach") or "").strip()
        new_a = (s.get("new_approach") or "").strip()
        reasoning = (s.get("reasoning") or "").strip()

        # keep compact
        def _cap(t: str, n: int = 240) -> str:
            t = t.replace("\n", " ").strip()
            return t[:n] + ("..." if len(t) > n else "")

        lines.append(f"[Skill {i}] {title}" + (f" (type={ctype})" if ctype else ""))
        if old_a:
            lines.append(f"- Old: {_cap(old_a)}")
        if new_a:
            lines.append(f"- New: {_cap(new_a)}")
        if reasoning:
            lines.append(f"- Why: {_cap(reasoning)}")
        lines.append("")

    return "\n".join(lines).strip()


def _safe_retrieve_skills(query: str, top_k: int = 6) -> List[Dict[str, Any]]:
    """
    Best-effort skill retrieval.
    Expects Cell 1b to define:
      retrieve_skills(query: str, top_k: int) -> List[Dict[str, Any]]
    If not present, returns [] without failing.
    """
    fn = globals().get("retrieve_skills", None)
    if not callable(fn):
        return []
    try:
        return fn(query=query, top_k=top_k) or []
    except Exception as e:
        log_info(f"[WARN] Skill retrieval failed (continuing without skills): {type(e).__name__}: {e}")
        return []


def _build_effective_spec_text(spec_text: str) -> str:
    """
    Build the final text passed to Claude (Cell 00 writer).
    - If Skill UI is available and enabled, retrieve and inject Skills as context.
    - Otherwise, return spec_text unchanged.
    """
    # If UI elements aren't defined, just return the original spec
    use_skill = bool(globals().get("use_skill_w").value) if "use_skill_w" in globals() else False
    if not use_skill:
        return spec_text

    query = (globals().get("skill_query_w").value or "").strip() if "skill_query_w" in globals() else ""
    top_k = int(globals().get("skill_topk_w").value) if "skill_topk_w" in globals() else 6
    if not query:
        query = "notebook generation structure pipeline"  # safe default

    skills = _safe_retrieve_skills(query=query, top_k=top_k)
    ctx = _format_skills_as_context(skills, max_items=top_k)

    if not ctx:
        return spec_text

    # Inject as preface (Claude tends to follow earlier constraints)
    return (
        "You MUST incorporate the following Skill Context as constraints/guardrails.\n\n"
        f"{ctx}\n\n"
        "## Requested Notebook Spec\n"
        f"{spec_text}"
    )


def on_generate_click(_):
    global _is_running
    global LAST_SPEC_TEXT, LAST_SAVED_NOTEBOOK_PATH

    if _is_running:
        return
    _is_running = True
    generate_btn.disabled = True

    with ui_out:
        clear_output()
        try:
            # 1) Read inputs
            spec_text = (spec_text_w.value or "").strip()
            outfile_path = (outfile_w.value or "").strip()

            if not spec_text:
                print("❌ SPEC_TEXT is empty. Paste the notebook specification first.")
                return
            if not outfile_path:
                print("❌ Outfile path is empty. Provide a destination .ipynb path.")
                return

            LAST_SPEC_TEXT = spec_text

            # 2) Build effective spec (optionally inject Skills)
            use_skill = bool(globals().get("use_skill_w").value) if "use_skill_w" in globals() else False
            print(f"📚 Skill integration: {'enabled' if use_skill else 'disabled'}")
            effective_spec_text = _build_effective_spec_text(spec_text)

            # 3) Claude generates Cell 00 (strict JSON)
            print("🧠 Calling Claude to generate Cell 00 (Structure source of truth)...")
            cell00_obj = call_claude_cell00(effective_spec_text)

            notebook_id = cell00_obj.get("notebook_id", "UNKNOWN_NOTEBOOK")
            cell00_source = cell00_obj.get("cell00_source", "")

            print(f"✅ Cell 00 received (notebook_id={notebook_id})")

            # Quick sanity check: must contain '# Structure' and '# Notes'
            if "# Structure" not in cell00_source or "# Notes" not in cell00_source:
                raise ValueError(
                    "Cell 00 is missing required section headers ('# Structure' / '# Notes'). "
                    "Please regenerate Cell 00."
                )

            # 4) Parse Structure items
            print("🔎 Parsing # Structure section...")
            items = parse_structure_items(cell00_source)

            # 5) Build skeleton notebook deterministically
            print("🧱 Building skeleton notebook (Cell 00 + Cell 01..N)...")
            nb = build_skeleton_notebook(cell00_source, items)

            # 6) Save
            saved_path = save_skeleton_notebook(nb, outfile_path)
            LAST_SAVED_NOTEBOOK_PATH = saved_path
            print(f"✅ Saved skeleton notebook: {saved_path}")

            # 7) Show parsed structure + preview
            _show_structure_items(items)

            if preview_w.value:
                display(Markdown("## Skeleton Notebook Preview"))
                display(Markdown(preview_saved_notebook(saved_path, n=int(preview_n_w.value))))

            # 8) Next steps
            display(Markdown(
                "**Next (Phase 2):** Fill cells incrementally via Claude patches (Cell 6a/6b).\n\n"
                "- Choose 1–2 target cells (e.g., Cell 02 / Cell 03)\n"
                "- Generate a patch JSON with Claude (Cell 6a)\n"
                "- Apply it deterministically to the saved .ipynb (Cell 6b)\n\n"
                "Reusable globals:\n"
                "- `LAST_SPEC_TEXT`\n"
                "- `LAST_CLAUDE_CELL00`\n"
                "- `LAST_STRUCTURE_ITEMS`\n"
                "- `LAST_SAVED_NOTEBOOK_PATH`"
            ))

        except Exception as e:
            print("❌ Skeleton generation failed.")
            print(type(e).__name__, ":", str(e))
            print("\n--- Traceback ---")
            traceback.print_exc()

            # Helpful debug: show Structure block if parse fails
            try:
                if "cell00_source" in locals() and cell00_source:
                    display(Markdown("## Debug: Extracted Structure Block (raw lines)"))
                    block = _find_structure_block_lines(cell00_source)
                    display(Markdown("```text\n" + "\n".join(block) + "\n```"))
            except Exception:
                pass

        finally:
            _is_running = False
            generate_btn.disabled = False


# Attach handler
generate_btn.on_click(on_generate_click)
print("Ready: paste SPEC_TEXT and click the generate button.")


Ready: paste SPEC_TEXT and click the generate button.


In [15]:
# ============================================================
# Cell 7 — Notebook Patch Generator (Phase 2: Fill Cells One-by-One)
# ============================================================
# Overview:
# Generates strict JSON patches to fill/replace specific cells in the generated target notebook.
# Claude is used ONLY at authoring time (this generator notebook).
#
# Patch JSON:
# {
#   "updates": [{"index": <int>, "source": "<FULL cell source>"}],
#   "notes": ["...optional..."]
# }
#
# Improvements vs prior:
# - Validates that updates target ONLY requested indices
# - Validates that each updated cell preserves the existing mandatory header prefix
# - Optional: inject Skill context (Cell 1b) into patch prompt for consistency
# - Adds safety caps to prompt size (cell source truncation)
#
# Notes:
# - Canonical headers MUST be preserved at the top of each cell.
# - Never introduce Claude/Anthropic runtime calls into the target notebook.
# - Prefer small patches (1 cell at a time) to reduce inconsistencies.
#

import json
import re
import nbformat
from pathlib import Path
from typing import Any, Dict, List

# ------------------------------------------------------------
# Notebook patch configuration
# ------------------------------------------------------------
NOTEBOOK_PATCH_MODEL = "claude-sonnet-4-5-20250929"
NOTEBOOK_PATCH_MAX_TOKENS = 4000

LAST_NOTEBOOK_PATCH = None

NOTEBOOK_PATCH_SYSTEM = """You are a precise Jupyter notebook cell writer.

Return ONLY valid JSON. No prose, no markdown fences, no backticks.
The JSON must be directly parseable by Python json.loads().

You must obey the target notebook's MANDATORY CONVENTIONS:
- Do NOT remove the mandatory cell header block at the top of each code cell.
- Keep cell numbering and titles consistent with the existing header.
- Keep code consistent with earlier cells (imports, variables, conventions).
- Do NOT add any Claude/Anthropic runtime calls into the target notebook.
- Prefer robust, readable code skeletons with clear TODOs.
- Avoid executing network calls during generation; write code that will run later in the target notebook.
- Keep each updated cell concise (target: <= 120 lines). Prefer TODO blocks over full implementations.
- Do not include long sample data, long explanations, or large embedded schemas; summarize as TODO.

Output schema:
{
  "updates": [
    {"index": <int>, "source": "<FULL replacement cell source string>"}
  ],
  "notes": ["...optional short notes..."]
}

Rules:
- Update ONLY the requested indices.
- Provide COMPLETE source for each updated cell (not a diff).
- Each 'source' must begin with the existing mandatory header for that cell.
"""

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def _extract_first_json_object(text: str) -> str:
    """
    Extract the first top-level JSON object from a string.
    Robust to accidental leading/trailing text.
    """
    text = (text or "").strip()
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object found in model output (missing '{').")

    depth = 0
    in_str = False
    esc = False
    end = None

    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is None:
        raise ValueError("JSON object appears incomplete (unmatched braces).")

    return text[start:end].strip()


def load_target_notebook(path: Path) -> nbformat.NotebookNode:
    return nbformat.read(str(path), as_version=4)


def _cap_text(s: str, max_chars: int) -> str:
    s = s or ""
    if len(s) <= max_chars:
        return s
    head = s[: int(max_chars * 0.6)]
    tail = s[-int(max_chars * 0.4) :]
    return head + "\n\n...[TRUNCATED]...\n\n" + tail


def _format_cells_for_prompt(
    nb: nbformat.NotebookNode,
    indices: List[int],
    label: str,
    *,
    per_cell_cap_chars: int = 9000
) -> str:
    parts: List[str] = []
    for i in indices:
        if i < 0 or i >= len(nb.cells):
            raise IndexError(f"Cell index out of range: {i} (cells={len(nb.cells)})")
        src = nb.cells[i].source or ""
        src = _cap_text(src, per_cell_cap_chars)
        parts.append(f"--- CELL {i:02d} ({label}) ---\n{src}\n")
    return "".join(parts)


def _get_existing_header_prefix(cell_source: str, *, max_lines: int = 40) -> str:
    """
    Extract a conservative "header prefix" from the existing skeleton cell:
    We take everything from start until the first blank line AFTER the header block,
    but cap to max_lines for safety.
    """
    lines = (cell_source or "").splitlines()
    if not lines:
        return ""
    prefix_lines = []
    blank_seen = False
    for line in lines[:max_lines]:
        prefix_lines.append(line)
        if line.strip() == "":
            # The skeleton header usually ends with a blank line
            blank_seen = True
            break
    # If no blank line, still return the first chunk; better than nothing
    return "\n".join(prefix_lines).rstrip() + "\n"


def _format_skills_as_context(skills: List[Dict[str, Any]], max_items: int = 6) -> str:
    """
    Compact skills context block (optional).
    """
    if not skills:
        return ""

    def _cap(t: str, n: int = 220) -> str:
        t = (t or "").replace("\n", " ").strip()
        return t[:n] + ("..." if len(t) > n else "")

    lines = []
    lines.append("## Skill Context (retrieved from Notebook Corrections DB)")
    lines.append("Apply these lessons as constraints/guardrails. Do not copy verbatim.\n")
    for i, s in enumerate(skills[:max_items], 1):
        title = _cap(s.get("title", "untitled"), 120)
        ctype = _cap(s.get("correction_type", ""), 40)
        old_a = _cap(s.get("old_approach", ""), 220)
        new_a = _cap(s.get("new_approach", ""), 220)
        why = _cap(s.get("reasoning", ""), 220)

        lines.append(f"[Skill {i}] {title}" + (f" (type={ctype})" if ctype else ""))
        if old_a:
            lines.append(f"- Old: {old_a}")
        if new_a:
            lines.append(f"- New: {new_a}")
        if why:
            lines.append(f"- Why: {why}")
        lines.append("")
    return "\n".join(lines).strip()


def _safe_retrieve_skills(query: str, top_k: int = 6) -> List[Dict[str, Any]]:
    """
    Best-effort retrieval. Requires Cell 1b to define retrieve_skills(query, top_k).
    """
    fn = globals().get("retrieve_skills", None)
    if not callable(fn):
        return []
    try:
        return fn(query=query, top_k=top_k) or []
    except Exception as e:
        log_info(f"[WARN] Skill retrieval failed (continuing without skills): {type(e).__name__}: {e}")
        return []


def build_patch_user_prompt(
    nb: nbformat.NotebookNode,
    target_indices: List[int],
    *,
    context_upto: int = 8,
    include_skills: bool = False,
    skill_query: str = "",
    skill_top_k: int = 6,
) -> str:
    """
    Prompt includes:
    - Optional Skill context
    - Context cells: Cell 00..context_upto (do not modify)
    - Target cells: existing skeleton content (must preserve header)
    """
    if not target_indices:
        raise ValueError("target_indices is empty")

    max_ctx = min(context_upto, len(nb.cells) - 1)
    ctx_indices = list(range(0, max_ctx + 1))

    ctx = _format_cells_for_prompt(nb, ctx_indices, "CONTEXT (do not modify)")
    tgt = _format_cells_for_prompt(nb, target_indices, "TARGET TO UPDATE (replace full source)")

    skill_block = ""
    if include_skills:
        q = (skill_query or "").strip() or "notebook cell patching conventions"
        skills = _safe_retrieve_skills(query=q, top_k=skill_top_k)
        skill_block = _format_skills_as_context(skills, max_items=skill_top_k)
        if skill_block:
            skill_block = skill_block + "\n\n"

    return f"""
You will fill specific cells in an existing notebook.

Requested target indices: {target_indices}

{skill_block}Context cells (do NOT modify; for consistency):
{ctx}

Target cells to update (replace FULL source; keep the mandatory header EXACTLY):
{tgt}
""".strip()

In [16]:
# ============================================================
# PATCH VALIDATION — relaxed header check (REPLACE THIS FUNCTION)
# ============================================================

import re
from typing import List, Dict, Any
# ------------------------------------------------------------
# Claude patch configuration (MUST be defined before functions)
# ------------------------------------------------------------
CLAUDE_PATCH_MODEL = "claude-sonnet-4-5-20250929"
CLAUDE_PATCH_MAX_TOKENS = 6500

def _validate_patch_against_targets(patch: Dict[str, Any], target_indices: List[int], nb) -> None:
    """
    Validate:
    - updates only include requested target_indices
    - each updated cell preserves mandatory header lines (not strict prefix match)
    - no Anthropic runtime calls
    """
    if "updates" not in patch or not isinstance(patch["updates"], list):
        raise ValueError("Patch JSON must contain an 'updates' list.")

    requested = set(int(i) for i in target_indices)
    got = set()

    def _normalize(s: str) -> str:
        return (s or "").replace("\r\n", "\n").replace("\r", "\n")

    def _extract_cell_title_line(existing_source: str) -> str:
        for ln in _normalize(existing_source).split("\n")[:30]:
            if ln.strip().startswith("# Cell "):
                return ln.strip()
        return ""

    def _required_header_lines(cell_title_line: str) -> List[str]:
        # These are your canonical mandatory header elements
        return [
            "# ============================================================",
            cell_title_line,
            "# ============================================================",
            "# Overview:",
            "#",
            "# Inputs / Outputs:",
            "#",
            "# Notes:",
        ]

    for upd in patch["updates"]:
        if not isinstance(upd, dict):
            raise ValueError("Each update must be an object.")
        if "index" not in upd or "source" not in upd:
            raise ValueError("Each update must include 'index' and 'source'.")

        idx = int(upd["index"])
        src = upd["source"]
        if not isinstance(src, str):
            raise ValueError("'source' must be a string.")
        got.add(idx)

        if idx not in requested:
            raise ValueError(
                f"Patch includes an update for index={idx}, but requested target_indices={sorted(requested)}"
            )
        if idx < 0 or idx >= len(nb.cells):
            raise IndexError(f"Cell index out of range: {idx} (cells={len(nb.cells)})")

        existing = nb.cells[idx].source or ""
        title_line = _extract_cell_title_line(existing)
        if not title_line:
            raise ValueError(f"Existing cell {idx} is missing a '# Cell XX — ...' header line.")

        required = _required_header_lines(title_line)

        # Check within top ~60 lines (header area)
        head = "\n".join([l.strip() for l in _normalize(src).split("\n")[:60]])

        missing = [r for r in required if r not in head]
        if missing:
            raise ValueError(
                f"Updated cell {idx} does not preserve the mandatory header lines.\n"
                f"Missing: {missing}\n"
                f"Expected to find them in the top section of the cell.\n"
                f"Got start:\n{_normalize(src)[:400]}"
            )

        # Prevent accidental Anthropic runtime usage
        if re.search(r"\banthropic\b|\bAnthropic\b|\bclaude_client\b", src):
            raise ValueError(
                f"Updated cell {idx} appears to include Anthropic/Claude runtime usage, which is disallowed."
            )

    # Ensure all requested indices are present (strict)
    if got != requested:
        raise ValueError(
            f"Patch updates indices mismatch.\nRequested: {sorted(requested)}\nGot: {sorted(got)}"
        )


# ------------------------------------------------------------
# Main API
# ------------------------------------------------------------
def call_claude_patch_for_cells(
    *,
    notebook_path: Path,
    target_indices: List[int],
    context_upto: int = 8,
    include_skills: bool = False,
    skill_query: str = "",
    skill_top_k: int = 6,
    max_tokens: int = CLAUDE_PATCH_MAX_TOKENS,
) -> Dict[str, Any]:
    """
    High-level helper:
    - loads the notebook
    - builds the prompt with context + targets (+ optional skills)
    - calls Claude
    - parses, validates, and returns the JSON patch
    """
    nb = load_target_notebook(notebook_path)
    user_prompt = build_patch_user_prompt(
        nb,
        target_indices,
        context_upto=context_upto,
        include_skills=include_skills,
        skill_query=skill_query,
        skill_top_k=skill_top_k,
    )

    resp = claude_client.messages.create(
        model=CLAUDE_PATCH_MODEL,
        system=CLAUDE_PATCH_SYSTEM,
        messages=[{"role": "user", "content": user_prompt}],
        max_tokens=int(max_tokens),
    )

    parts = []
    for block in resp.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
        else:
            parts.append(str(block))
    raw = "\n".join(parts).strip()

    json_str = _extract_first_json_object(raw)
    patch = json.loads(json_str)

    # Normalize indices
    for upd in patch.get("updates", []):
        if isinstance(upd, dict) and "index" in upd:
            upd["index"] = int(upd["index"])

    # Validate (strict)
    _validate_patch_against_targets(patch, target_indices=target_indices, nb=nb)

    global LAST_CLAUDE_PATCH
    LAST_CLAUDE_PATCH = patch
    return patch


# Optional "tight" variant for shorter output
CELL9_EXTRA_RULES = """
Extra constraints for this patch:
- Keep the cell very concise (target <= 80 lines).
- Do NOT implement full production logic. Provide a readable skeleton with TODOs.
- No long try/except ladders. No long docstrings. No large embedded schemas.
- Prefer small helper functions and clear TODO steps.
- The result MUST be valid JSON and must close all braces/brackets.
"""

def call_claude_patch_for_cells_tight(
    *,
    notebook_path: Path,
    target_indices: List[int],
    context_upto: int = 10,
    include_skills: bool = False,
    skill_query: str = "",
    skill_top_k: int = 6,
) -> Dict[str, Any]:
    nb = load_target_notebook(notebook_path)
    user_prompt = build_patch_user_prompt(
        nb,
        target_indices,
        context_upto=context_upto,
        include_skills=include_skills,
        skill_query=skill_query,
        skill_top_k=skill_top_k,
    )
    user_prompt = user_prompt + "\n\n" + CELL9_EXTRA_RULES

    resp = claude_client.messages.create(
        model=CLAUDE_PATCH_MODEL,
        system=CLAUDE_PATCH_SYSTEM,
        messages=[{"role": "user", "content": user_prompt}],
        max_tokens=2500,  # shorter output
    )

    parts = []
    for block in resp.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
        else:
            parts.append(str(block))
    raw = "\n".join(parts).strip()

    json_str = _extract_first_json_object(raw)
    patch = json.loads(json_str)

    for upd in patch.get("updates", []):
        if isinstance(upd, dict) and "index" in upd:
            upd["index"] = int(upd["index"])

    _validate_patch_against_targets(patch, target_indices=target_indices, nb=nb)

    global LAST_CLAUDE_PATCH
    LAST_CLAUDE_PATCH = patch
    return patch


# ------------------------------------------------------------
# Example usage (uncomment and run)
# ------------------------------------------------------------
# target_nb_path = LAST_SAVED_NOTEBOOK_PATH
# patch = call_claude_patch_for_cells(
#     notebook_path=target_nb_path,
#     target_indices=[2],    # fill 1 cell at a time
#     context_upto=10,
#     include_skills=True,   # requires Cell 1b; otherwise auto-fallback to no skills
#     skill_query="notebook generator reliability json schema parsing",
#     skill_top_k=6,
# )
# print("Patch keys:", patch.keys())
# print("Updates:", [u["index"] for u in patch["updates"]])


In [17]:
# ============================================================
# Cell 6b — Patch Applier (Apply JSON Updates to .ipynb)
# ============================================================
# Overview:
# Applies a JSON patch (from Cell 6a) to the saved target notebook (.ipynb).
# Replaces the FULL `source` of specified cell indices deterministically, then saves in-place.
#
# Improvements vs prior:
# - Adds missing typing imports (List, Dict, Any)
# - Stronger + clearer header preservation enforcement:
#     - requires updated cell to start with the existing header prefix chunk
#     - optional secondary "required lines" check for robustness
# - Prevents duplicate / conflicting indices in a single patch
# - Safer normalization (strip + trailing newline) and clearer errors
#
# Notes:
# - No LLM calls here (purely deterministic).
# - Apply patches in small batches (1–2 cells).
#

import nbformat
from pathlib import Path
from typing import Any, Dict, List, Optional, Set

LAST_APPLIED_PATCH: Optional[Dict[str, Any]] = None

# ------------------------------------------------------------
# IO helpers
# ------------------------------------------------------------
def load_notebook(path: Path) -> nbformat.NotebookNode:
    return nbformat.read(str(path), as_version=4)

def save_notebook_inplace(nb: nbformat.NotebookNode, path: Path) -> None:
    nbformat.write(nb, str(path))

def _normalize_newlines(s: str) -> str:
    return (s or "").replace("\r\n", "\n").replace("\r", "\n")

# ------------------------------------------------------------
# Header checks
# ------------------------------------------------------------
def _existing_header_prefix(cell_source: str, *, max_lines: int = 40) -> str:
    """
    Extract a conservative "header prefix" from the existing cell:
    Take lines from the beginning until the first blank line (inclusive),
    capped to max_lines. This matches the generator's header block well.
    """
    s = _normalize_newlines(cell_source or "")
    lines = s.split("\n")
    prefix: List[str] = []
    for ln in lines[:max_lines]:
        prefix.append(ln)
        if ln.strip() == "":
            break
    return "\n".join(prefix).rstrip() + "\n"

def _extract_cell_title_line(cell_source: str) -> Optional[str]:
    """
    Find canonical "# Cell XX — ..." line from the existing cell.
    """
    lines = _normalize_newlines(cell_source or "").split("\n")
    for ln in lines[:60]:
        t = ln.strip()
        if t.startswith("# Cell "):
            return t
    return None

def _required_header_lines(cell_title_line: str) -> List[str]:
    """
    Minimal required header markers.
    We do not enforce exact blank line placement, but we require these tokens
    to exist near the top of the updated cell.
    """
    return [
        "# ============================================================",
        cell_title_line,
        "# ============================================================",
        "# Overview:",
        "# Inputs / Outputs:",
        "# Notes:",
    ]

def _header_ok(existing_source: str, new_source: str) -> bool:
    """
    Two-level check:
    1) Strong check: new_source must start with the existing header prefix chunk.
    2) Backup check: required header lines must appear in the top area.
    """
    existing_source = _normalize_newlines(existing_source or "")
    new_source = _normalize_newlines(new_source or "")

    prefix = _existing_header_prefix(existing_source)
    if prefix.strip():  # strong check when we have a usable prefix
        if new_source.startswith(prefix):
            return True

    # fallback: required markers in head
    title_line = _extract_cell_title_line(existing_source)
    if not title_line:
        return False

    required = _required_header_lines(title_line)
    head = "\n".join([l.strip() for l in new_source.split("\n")[:80]])
    return all(req in head for req in required)


In [18]:
# ------------------------------------------------------------
# Patch application
# ------------------------------------------------------------
def apply_patch_to_notebook(
    notebook_path: Path,
    patch: Dict[str, Any],
    *,
    enforce_header_prefix: bool = True,
) -> Path:
    """
    Apply patch updates to the notebook at notebook_path and save in-place.
    Returns the notebook path.
    """
    nb = load_notebook(notebook_path)

    updates = patch.get("updates")
    if not isinstance(updates, list) or not updates:
        raise ValueError("Patch must include a non-empty 'updates' list.")

    # Normalize + validate indices / sources
    norm_updates: List[Dict[str, Any]] = []
    seen: Set[int] = set()

    for upd in updates:
        if not isinstance(upd, dict):
            raise ValueError("Each update must be an object with 'index' and 'source'.")

        if "index" not in upd or "source" not in upd:
            raise ValueError("Each update must include 'index' and 'source'.")

        idx = int(upd["index"])
        if idx in seen:
            raise ValueError(f"Duplicate update index in patch: {idx}")
        seen.add(idx)

        src_new = upd["source"]
        if not isinstance(src_new, str):
            raise ValueError(f"Update 'source' must be a string (index={idx}).")

        norm_updates.append({"index": idx, "source": src_new})

    # Apply in ascending order (deterministic)
    for upd in sorted(norm_updates, key=lambda u: u["index"]):
        idx = upd["index"]
        src_new = upd["source"]

        if idx < 0 or idx >= len(nb.cells):
            raise IndexError(f"Cell index out of range: {idx} (cells={len(nb.cells)})")

        src_new = _normalize_newlines(src_new).strip() + "\n"

        if enforce_header_prefix:
            if not _header_ok(nb.cells[idx].source or "", src_new):
                prefix = _existing_header_prefix(nb.cells[idx].source or "")
                raise ValueError(
                    "Header preservation check failed.\n"
                    f"Cell index: {idx}\n\n"
                    "Expected the updated cell to preserve the mandatory header.\n"
                    "Tip: ensure the update 'source' begins with the existing header block.\n\n"
                    f"--- Expected header prefix (from existing cell) ---\n{prefix}\n"
                    f"--- Updated cell starts with ---\n{src_new[:300]}"
                )

        nb.cells[idx].source = src_new

    save_notebook_inplace(nb, notebook_path)

    global LAST_APPLIED_PATCH
    LAST_APPLIED_PATCH = patch
    return notebook_path

# ------------------------------------------------------------
# Preview helper
# ------------------------------------------------------------
def preview_notebook(path: Path, n: int = 12) -> str:
    """
    Markdown preview of the first N cells (reads from disk).
    """
    nb = nbformat.read(str(path), as_version=4)
    lines: List[str] = []
    lines.append(f"**Saved file:** `{path}`")
    lines.append(f"**Cells:** {len(nb.cells)}\n")

    for i, c in enumerate(nb.cells[:n]):
        src = (c.source or "").strip()
        head = src[:700] + ("..." if len(src) > 700 else "")
        lines.append(f"### Cell {i:02d} ({c.cell_type})")
        lines.append(f"```python\n{head}\n```")

    return "\n".join(lines)

# ------------------------------------------------------------
# Example usage (uncomment and run)
# ------------------------------------------------------------
# target_nb_path = LAST_SAVED_NOTEBOOK_PATH
# patch = LAST_CLAUDE_PATCH
# apply_patch_to_notebook(target_nb_path, patch, enforce_header_prefix=True)
# from IPython.display import Markdown, display
# display(Markdown(preview_notebook(target_nb_path, n=12)))


In [19]:
# ===================================================================================
# Cell 7 — Fill All Cells (Robust Batch Runner)
# ===================================================================================
# Overview:# Fill all target notebook cells via Claude patches with automatic fallback:
#   normal batch → single-cell → tight mode (short & constrained).
#
# Improvements vs prior:
# - Adds missing imports (List, Markdown, display)
# - Safer handling when LAST_SAVED_NOTEBOOK_PATH is None
# - Better error messages + progress reporting
# - Header-check failures are treated as "recoverable" (fallback) rather than crashing early
# - Optional "continue_on_failure" to avoid stopping the whole run on one hard cell
#
# Notes:
# - JSON failures are expected; this runner degrades gracefully.
# - A single cell failure does NOT corrupt previously written cells.
# - This cell assumes you already defined:
#     - call_claude_patch_for_cells
#     - call_claude_patch_for_cells_tight
#     - apply_patch_to_notebook
#     - preview_notebook
#

import nbformat
from pathlib import Path
from typing import List, Optional, Dict, Any

from IPython.display import display, Markdown
# ------------------------------------------------------------
# Claude patch configuration (SYSTEM PROMPT)
# ------------------------------------------------------------
CLAUDE_PATCH_SYSTEM = r"""
You are an expert Jupyter Notebook author.

You must return ONLY a strict JSON object:
{
  "updates": [
    { "index": <int>, "source": "<FULL cell source>" }
  ]
}

HARD RULES (do not violate):
1) NO text outside JSON. No markdown. No explanations.
2) Update ONLY the requested cell indices.
3) The updated cell MUST preserve the existing mandatory header EXACTLY.
   - You are given the existing cell header in the prompt context.
   - Copy/paste the header block verbatim as the first lines of "source".
   - Do NOT rename the cell title.
   - Do NOT alter Overview/Inputs/Notes header lines.
4) Add all new code ONLY below the preserved header block.
5) Do NOT include Anthropic/Claude runtime usage inside the generated notebook code.
6) Ensure valid JSON (all braces/brackets closed) and valid Python.

If you are repairing a failing cell:
- Make the minimal change that fixes the error.
- Prefer adding missing constants/imports to Cell 01 rather than moving logic across cells.
"""

# ----------------------------------------------------------
# Utilities
# ----------------------------------------------------------
def get_num_cells(path: Path) -> int:
    nb = nbformat.read(str(path), as_version=4)
    return len(nb.cells)

def build_batches(total_cells: int) -> List[List[int]]:
    """
    Default conservative batching strategy.
    Cell 0 (Cell 00) is excluded.
    """
    indices = list(range(1, total_cells))

    singles = indices[:5]      # 1..5
    rest = indices[5:]         # 6..

    batches: List[List[int]] = [[i] for i in singles]

    # Escalate slowly
    chunk_sizes = [2, 2, 3, 3]
    pos, k = 0, 0
    while pos < len(rest):
        size = chunk_sizes[min(k, len(chunk_sizes) - 1)]
        batches.append(rest[pos : pos + size])
        pos += size
        k += 1

    return batches

# ----------------------------------------------------------
# Patch attempt helpers
# ----------------------------------------------------------
_RECOVERABLE_MARKERS = [
    # JSON extraction/parse failures
    "JSON object appears incomplete",
    "No JSON object found",
    "Invalid \\u",
    "Unterminated string",
    "Expecting value",
    "Extra data",
    # Header enforcement failures (recoverable by trying tight/single)
    "Header preservation check failed",
]

def _is_recoverable_error(e: Exception) -> bool:
    msg = str(e) or ""
    return any(m in msg for m in _RECOVERABLE_MARKERS)

def try_patch(
    *,
    notebook_path: Path,
    targets: List[int],
    context_upto: int,
    enforce_header_prefix: bool,
    mode: str = "normal",
) -> bool:
    """
    mode:
      - "normal": call_claude_patch_for_cells
      - "tight":  call_claude_patch_for_cells_tight

    Returns True if patch applied successfully.
    Returns False for recoverable errors (JSON/header issues).
    Raises for non-recoverable errors (real bugs).
    """
    try:
        if mode == "normal":
            patch = call_claude_patch_for_cells(
                notebook_path=notebook_path,
                target_indices=targets,
                context_upto=context_upto,
            )
        elif mode == "tight":
            patch = call_claude_patch_for_cells_tight(
                notebook_path=notebook_path,
                target_indices=targets,
                context_upto=context_upto,
            )
        else:
            raise ValueError(f"Unknown mode: {mode}")

        apply_patch_to_notebook(
            notebook_path,
            patch,
            enforce_header_prefix=enforce_header_prefix,
        )
        return True

    except Exception as e:
        if _is_recoverable_error(e):
            return False
        raise  # non-recoverable → surface it

# ----------------------------------------------------------
# Main runner
# ----------------------------------------------------------
def fill_all_cells_in_batches(
    notebook_path: Path,
    *,
    context_upto: int = 12,
    enforce_header_prefix: bool = True,
    preview_every_batch: bool = False,
    preview_n: int = 8,
    continue_on_failure: bool = False,
) -> Dict[str, Any]:
    """
    Run batch fill across all cells.
    Returns a stats dict for inspection.
    """
    if notebook_path is None:
        raise ValueError("notebook_path is None. Generate a skeleton first (Cell 5).")

    notebook_path = Path(notebook_path).expanduser().resolve()
    if not notebook_path.exists():
        raise FileNotFoundError(f"Notebook not found: {notebook_path}")

    total = get_num_cells(notebook_path)
    print(f"📘 Target notebook: {notebook_path}")
    print(f"🔢 Total cells: {total}")

    batches = build_batches(total)
    print(f"🧩 Batches: {len(batches)}")
    print("Batch plan:", batches)

    stats = {
        "total_cells": total,
        "batches": len(batches),
        "patched_cells": set(),
        "failed_cells": [],
        "mode_success_counts": {"normal_batch": 0, "normal_single": 0, "tight_single": 0},
    }

    for bi, targets in enumerate(batches, start=1):
        print("\n" + "=" * 72)
        print(f"🚆 Batch {bi}/{len(batches)} — targets={targets}")

        # 1) Normal batch attempt
        ok = try_patch(
            notebook_path=notebook_path,
            targets=targets,
            context_upto=context_upto,
            enforce_header_prefix=enforce_header_prefix,
            mode="normal",
        )
        if ok:
            print("✅ Patch applied (normal batch mode).")
            stats["mode_success_counts"]["normal_batch"] += 1
            for t in targets:
                stats["patched_cells"].add(t)
            if preview_every_batch:
                display(Markdown(preview_notebook(notebook_path, n=preview_n)))
            continue

        print("⚠️ Normal batch mode failed. Falling back to single-cell mode.")

        # 2) Single-cell fallback
        for t in targets:
            if t in stats["patched_cells"]:
                continue

            print(f"   ↪ Cell [{t}] — normal single attempt")
            ok_single = try_patch(
                notebook_path=notebook_path,
                targets=[t],
                context_upto=context_upto,
                enforce_header_prefix=enforce_header_prefix,
                mode="normal",
            )
            if ok_single:
                print(f"   ✅ Cell [{t}] patched (normal single).")
                stats["mode_success_counts"]["normal_single"] += 1
                stats["patched_cells"].add(t)
                continue

            # 3) Tight mode fallback
            print(f"   ↪ Cell [{t}] ℔ tight single attempt")
            ok_tight = try_patch(
                notebook_path=notebook_path,
                targets=[t],
                context_upto=context_upto,
                enforce_header_prefix=enforce_header_prefix,
                mode="tight",
            )
            if ok_tight:
                print(f"   ✅ Cell [{t}] patched (tight single).")
                stats["mode_success_counts"]["tight_single"] += 1
                stats["patched_cells"].add(t)
                continue

            # Unrecoverable for this cell (after fallbacks)
            msg = (
                f"   ❌ Cell [{t}] FAILED even in tight mode.\n"
                "      → Manual inspection recommended."
            )
            print(msg)
            stats["failed_cells"].append(t)

            if not continue_on_failure:
                raise RuntimeError(f"Unrecoverable failure for cell [{t}]")

        if preview_every_batch:
            display(Markdown(preview_notebook(notebook_path, n=preview_n)))

    print("\n⟏ Done. Batch runner completed.")
    if stats["failed_cells"]:
        print(f"⚠️ Failed cells: {stats['failed_cells']}")
    else:
        print("✅ All cells processed successfully.")

    display(Markdown(preview_notebook(notebook_path, n=preview_n)))
    return stats


# ----------------------------------------------------------
# RUN
# ----------------------------------------------------------
target_nb_path = LAST_SAVED_NOTEBOOK_PATH
stats = fill_all_cells_in_batches(
    notebook_path=target_nb_path,
    context_upto=14,
    enforce_header_prefix=True,
    preview_every_batch=False,
    preview_n=10,
    continue_on_failure=False,  # set True if you want to finish even with some failures
)
print("Run stats:", {k: (len(v) if isinstance(v, set) else v) for k, v in stats.items()})


📘 Target notebook: /Users/yuetoya/projects/researchOS100-private/notebooks/041_weekly_events_digest.ipynb
🔢 Total cells: 12
🧩 Batches: 8
Batch plan: [[1], [2], [3], [4], [5], [6, 7], [8, 9], [10, 11]]

🚆 Batch 1/8 — targets=[1]
✅ Patch applied (normal batch mode).

🚆 Batch 2/8 — targets=[2]
✅ Patch applied (normal batch mode).

🚆 Batch 3/8 — targets=[3]
✅ Patch applied (normal batch mode).

🚆 Batch 4/8 — targets=[4]
✅ Patch applied (normal batch mode).

🚆 Batch 5/8 — targets=[5]
✅ Patch applied (normal batch mode).

🚆 Batch 6/8 — targets=[6, 7]
✅ Patch applied (normal batch mode).

🚆 Batch 7/8 — targets=[8, 9]
✅ Patch applied (normal batch mode).

🚆 Batch 8/8 — targets=[10, 11]
✅ Patch applied (normal batch mode).

⟏ Done. Batch runner completed.
✅ All cells processed successfully.


**Saved file:** `/Users/yuetoya/projects/researchOS100-private/notebooks/041_weekly_events_digest.ipynb`
**Cells:** 12

### Cell 00 (code)
```python
# ============================================================
# 041_weekly_events_digest
# ============================================================
#
# Overview
# ----------------
# Weekly aggregation notebook that queries the last 7 days of Events
# from Notion, deduplicates, clusters into themes, ranks signals,
# and writes a weekly digest back to Notion + exports local artifacts.
# Runs in Asia/Tokyo timezone with configurable noise filters.
# Optionally uses LLM (gpt-4o-mini) for theme labeling.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - NOTION_EVENTS_DB_ID: Notion data source containing all events
#   - NOTION_WEEKLY_DIGESTS_DS_ID: Target data source for digest pages
#...
```
### Cell 01 (code)
```python
# ============================================================
# Cell 01 — Imports, environment setup, constants, logging, run_id
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Imports ---
import os
import sys
import json
import logging
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, List, Any, Optional, Set
from collections import defaultdict
import hashlib
import re

#...
```
### Cell 02 (code)
```python
# ============================================================
# Cell 02 — Notion API wrappers and schema helpers
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# --- Notion API headers ---
notion_headers = {
    'Authorization': f'Bearer {NOTION_TOKEN}',
    'Notion-Version': NOTION_API_VERSION,
    'Content-Type': 'application/json'
}

# --- Schema property extraction helpers ---

def get_property_value(properties: Dict, prop_name: str, prop_type: str, default: Any = None) -> Any:
    """
    Extract a property value from Notion page properties dict.
    
    Args:
        properties: Notion page properties dict
        prop_n...
```
### Cell 03 (code)
```python
# ============================================================
# Cell 03 — Time window computation (last 7 days, JST)
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Compute the time window for weekly digest (last 7 days)
now_jst = datetime.now(TIMEZONE)
week_end = now_jst
week_start = week_end - timedelta(days=7)

# Format for logging and digest title
week_start_str = week_start.strftime('%Y-%m-%d')
week_end_str = week_end.strftime('%Y-%m-%d')
week_label = f"{week_start.strftime('%Y-W%V')}"  # ISO week format

logger.info(f"Time window: {week_start_str} to {week_end_str} ({TIMEZONE})")
logger.info(f"Week label: {week_label}")
...
```
### Cell 04 (code)
```python
# ============================================================
# Cell 04 — Fetch weekly events from Events data source
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Build Notion filter for events in the time window
# Filter by Event Date property within [week_start, week_end]
event_filter = {
    'and': [
        {
            'property': 'Event Date',
            'date': {
                'on_or_after': week_start_iso
            }
        },
        {
            'property': 'Event Date',
            'date': {
                'on_or_before': week_end_iso
            }
        }
    ]
}

# Optional: Sort by Event Date descen...
```
### Cell 05 (code)
```python
# ============================================================
# Cell 05 — Normalize records and validate fields
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Normalize each fetched event into a structured dict with validated fields
normalized_events = []

for page in fetched_events:
    page_id = page.get('id')
    props = page.get('properties', {})
    
    # Extract core fields using schema helpers
    event_title = get_property_value(props, 'Title', 'title', default='Untitled Event')
    event_date = get_property_value(props, 'Event Date', 'date', default=None)
    event_type = get_property_value(props, 'Event Type', 'sel...
```
### Cell 06 (code)
```python
# ============================================================
# Cell 06 — Deduplication and noise filtering
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Apply deduplication using the dedup_key field
# and filter out noise based on configured thresholds

seen_dedup_keys: Set[str] = set()
filtered_events = []

for event in normalized_events:
    dedup_key = event['dedup_key']
    confidence = event['confidence']
    status = event['status']
    source = event['source']
    
    # Skip duplicates (first occurrence wins)
    if dedup_key in seen_dedup_keys:
        logger.debug(f"Skipping duplicate event: {event['title']} (dedu...
```
### Cell 07 (code)
```python
# ============================================================
# Cell 07 — Theme clustering and optional LLM labeling
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Cluster filtered events into themes based on:
# 1. Target overlap (shared targets)
# 2. Keyword overlap
# 3. Event type similarity
# 4. Domain similarity
#
# Optionally use LLM to generate descriptive theme labels

from typing import List, Dict, Set, Tuple

def compute_overlap_score(set_a: Set[str], set_b: Set[str]) -> float:
    """Jaccard similarity between two sets."""
    if not set_a and not set_b:
        return 0.0
    intersection = len(set_a & set_b)
    u...
```
### Cell 08 (code)
```python
# ============================================================
# Cell 08 — Theme scoring and top N selection
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Score each theme and select top N most significant themes
# Scoring factors:
# 1. Event count (more events = higher score)
# 2. Average confidence (higher confidence = higher score)
# 3. Target diversity (more unique targets = higher score)
# 4. Domain diversity (multiple domains = higher score)
# 5. Recency (more recent events = higher score)

import statistics
from datetime import datetime

def score_theme(theme: Dict) -> float:
    """
    Compute a composite score for a...
```
### Cell 09 (code)
```python
# ============================================================
# Cell 09 — Render weekly digest markdown
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

# Render a markdown weekly digest from the top themes
# Format:
# - Header with week range and summary stats
# - Executive summary
# - Top N themes, each with:
#   - Theme label and metadata
#   - Key events (title, date, type, confidence)
#   - Related targets and keywords

from typing import List

def render_event_row(event: Dict) -> str:
    """
    Render a single event as a markdown list item.
    
    Format:
    - **[Title](URL)** (Type) - Date [Confidence: X.XX]
      Des...
```

Run stats: {'total_cells': 12, 'batches': 8, 'patched_cells': 11, 'failed_cells': [], 'mode_success_counts': {'normal_batch': 8, 'normal_single': 0, 'tight_single': 0}}


In [23]:
###############################################################################
## Cell 08: Phase 2 – Iterative Build-and-Test Loop for Cells 01..N        ##
##############################################################################

import nbformat
from nbformat.v4 import new_notebook, new_code_cell, new_markdown_cell
from nbclient import NotebookClient
import json
import re
import os
from openai import OpenAI


# Initialize OpenAI client
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY environment variable not set")
openai_client = OpenAI(api_key=api_key)

def execute_notebook(notebook_path):
    """
    Execute notebook via nbclient and return:
      (success, error_info)
    error_info contains failing cell index and traceback when possible.
    """
    with open(notebook_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    client = NotebookClient(nb, timeout=600, kernel_name="python3", allow_errors=True)

    try:
        client.execute()
    except CellExecutionError:
        # allow_errors=Trueなので、ここで落ちても outputs に error が入っていることが多い
        pass
    except Exception as e:
        return False, {
            "exception_type": type(e).__name__,
            "exception_message": str(e),
            "traceback": str(e),
            "failing_cell": None,
        }

    # Scan outputs to find first error cell
    for i, cell in enumerate(nb.cells):
        if cell.get("cell_type") != "code":
            continue
        for out in cell.get("outputs", []) or []:
            if out.get("output_type") == "error":
                tb = "\n".join(out.get("traceback", []) or [])
                return False, {
                    "exception_type": out.get("ename"),
                    "exception_message": out.get("evalue"),
                    "traceback": tb,
                    "failing_cell": i,
                }

    return True, None

def generate_repair_prompt_with_openai(cell00_content, cell_goal, current_source, error_info):
    """
    Use OpenAI to generate a repair instruction prompt for Claude.
    """
    system_prompt = """You are an expert at debugging Jupyter notebooks. Given a failing cell and error details, generate a clear, concise prompt that instructs Claude how to fix the cell."""
    
    user_prompt = f"""
Context:
Cell 00 contains the global structure and configuration:
{{cell00_content}}

Cell Goal:
{cell_goal}

Current Failing Cell Source:
{current_source}

Error Details:
- Exception Type: {error_info['exception_type']}
- Exception Message: {error_info['exception_message']}
- Traceback: {error_info['traceback']}

Constraints:
- Preserve the mandatory cell header block
- Keep code concise
- No Claude runtime calls in the generated notebook
- Fix the error directly
- The rewritten cell MUST start with the exact same header block as the current cell (copy verbatim).
- Do NOT rename the cell title line.

Generate a single prompt that Claude can use to fix this cell.
    """
    
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"WARNING: OpenAI repair prompt generation failed: {e}")
        # Fallback to a simple repair prompt
        return f"""Fix the following cell that is failing with error: {error_info['exception_type']}: {error_info['exception_message']}. Preserve the header block and keep code concise."""

def build_and_test_loop(
    target_notebook_path,
    cell00_content,
    structure_lines,
    max_fix_attempts=3,
    *,
    context_upto=14,
    openai_model="gpt-4o-mini",
    use_tight_on_json_failure=True,
):
    """
    Iteratively build and test each cell from 01..N using REAL Claude patch functions.

    Flow:
      - For each planned cell:
          1) Generate patch for that cell via Claude (normal; fallback to tight on JSON failure)
          2) Apply patch to notebook deterministically (apply_patch_to_notebook)
          3) Execute the notebook to detect the failing cell (execute_notebook must return failing_cell index)
          4) If error:
               - Build repair prompt with OpenAI using:
                   Cell 00 + failing cell goal + current failing cell source + error details
               - Send that prompt to Claude to generate a patch for the FAILING cell
               - Apply patch and re-execute
          5) Proceed when execution succeeds.
    """

    # ---- Parse structure goals ----
    cell_goals = {}
    for line in structure_lines:
        m = re.match(r'Cell (\d+): (.+)', line.strip())
        if m:
            cell_goals[int(m.group(1))] = m.group(2).strip()

    planned_cells = sorted([i for i in cell_goals.keys() if i > 0])

    def _read_nb(path):
        with open(path, "r") as f:
            return nbformat.read(f, as_version=4)

    def _apply_claude_patch_for_cell(cell_index: int, *, tight: bool = False, repair_user_prompt: str | None = None):
        """
        Apply a patch to a single cell.
        If repair_user_prompt is provided, we call Claude directly with that user prompt and parse JSON patch.
        Otherwise, we use call_claude_patch_for_cells / tight variant.
        """
        if repair_user_prompt is not None:
            # Call Claude directly with repair_user_prompt and parse JSON patch
            resp = claude_client.messages.create(
                model=CLAUDE_PATCH_MODEL,
                system=CLAUDE_PATCH_SYSTEM,
                messages=[{"role": "user", "content": repair_user_prompt}],
                max_tokens=int(CLAUDE_PATCH_MAX_TOKENS),
            )
            parts = []
            for block in resp.content:
                if getattr(block, "type", None) == "text":
                    parts.append(block.text)
                else:
                    parts.append(str(block))
            raw = "\n".join(parts).strip()

            json_str = _extract_first_json_object(raw)
            patch = json.loads(json_str)
        else:
            # Normal patch generation path
            if not tight:
                patch = call_claude_patch_for_cells(
                    notebook_path=Path(target_notebook_path),
                    target_indices=[cell_index],
                    context_upto=context_upto,
                )
            else:
                patch = call_claude_patch_for_cells_tight(
                    notebook_path=Path(target_notebook_path),
                    target_indices=[cell_index],
                    context_upto=context_upto,
                )

        # Apply patch deterministically
        apply_patch_to_notebook(
            Path(target_notebook_path),
            patch,
            enforce_header_prefix=True,
        )
        return patch

    print("🧪 Phase 2 — Iterative build-and-test starting...")
    print(f"Target notebook: {target_notebook_path}")
    print(f"Planned cells: {planned_cells}")

    for cell_idx in planned_cells:
        print("\n" + "=" * 66)
        print(f"🏗️ Processing Cell {cell_idx:02}: {cell_goals.get(cell_idx,'')}")
        print("=" * 66)

        # 1) Generate initial patch for THIS planned cell
        print("[Step] Generating initial patch...")
        try:
            _apply_claude_patch_for_cell(cell_idx, tight=False)
            print(f"✅ Patched Cell {cell_idx:02} (normal).")
        except Exception as e:
            msg = str(e)
            if use_tight_on_json_failure and ("JSON object appears incomplete" in msg or "No JSON object found" in msg):
                _apply_claude_patch_for_cell(cell_idx, tight=True)
                print(f"✅ Patched Cell {cell_idx:02} (tight fallback).")
            else:
                raise

        # 2) Execute + repair loop
        for attempt in range(1, max_fix_attempts + 1):
            print("[Step] Executing notebook...")
            success, error_info = execute_notebook(target_notebook_path)

            if success:
                print(f"✅ Notebook executes clean after Cell {cell_idx:02}. Proceeding.")
                break

            failing = (error_info or {}).get("failing_cell")
            etype = (error_info or {}).get("exception_type")
            emsg = (error_info or {}).get("exception_message")

            print(f"❌ Execution failed. failing_cell={failing}")
            print(f"   {etype}: {emsg}")

            if failing is None:
                print("⚠️ failing_cell is None. Your execute_notebook() must return a valid failing cell index.")
                return False

            if attempt == max_fix_attempts:
                print("❌ Max fix attempts reached. Stopping.")
                print(f"Last failing cell: {failing}")
                return False

            # 3) Build repair prompt with OpenAI for THE FAILING CELL
            nb_now = _read_nb(target_notebook_path)
            failing_src = nb_now.cells[failing].source if failing < len(nb_now.cells) else ""
            failing_goal = cell_goals.get(
                failing,
                f"Fix Cell {failing:02} so the notebook executes without errors."
            )

            print("🛠️ Generating repair prompt with OpenAI...")
            repair_user_prompt = generate_repair_prompt_with_openai(
                cell00_content=cell00_content,
                cell_goal=failing_goal,
                current_source=failing_src,
                error_info=error_info,
            )
            print("Repair prompt (head):", (repair_user_prompt or "")[:200], "...")

            # 4) Apply Claude repair patch to the failing cell
            print(f"[Step] Applying repair patch to failing Cell {failing:02} ...")
            try:
                _apply_claude_patch_for_cell(failing, repair_user_prompt=repair_user_prompt)
                print(f"✅ Repaired Cell {failing:02}.")
            except Exception as e:
                msg = str(e)
                if use_tight_on_json_failure and ("JSON object appears incomplete" in msg or "No JSON object found" in msg):
                    # If JSON broken even in repair, retry with tight normal generation
                    _apply_claude_patch_for_cell(failing, tight=True)
                    print(f"✅ Repaired Cell {failing:02} (tight fallback).")
                else:
                    raise

    print("\n🎉 All planned cells generated and notebook executes clean.")
    return True



def generate_cell_with_claude(cell_idx, cell_goal, repair_prompt=None):
    """
    Placeholder for Claude cell generation.
    This should call your existing Claude API logic.
    """
    # TODO: Implement actual Claude API call
    return f"# Cell {cell_idx:02}: Placeholder\nprint('Cell {cell_idx} generated')"

def apply_cell_patch(target_notebook_path, cell_index, cell_source):
    with open(target_notebook_path, 'r') as f:
        nb = nbformat.read(f, as_version=4)

    nb.cells[cell_index].source = cell_source

    with open(target_notebook_path, 'w') as f:
        nbformat.write(nb, f)

import nbformat
from nbclient import NotebookClient
from pathlib import Path

def execute_notebook_prefix_with_outputs(notebook_path: str | Path, upto_index: int, timeout: int = 600):
    """
    Execute cells 0..upto_index (inclusive) in a fresh kernel.
    Returns (success, info)

    info = {
      "failing_cell": int|None,
      "exception_type": str|None,
      "exception_message": str|None,
      "traceback": str|None,
      "cell_outputs": List[Dict]  # per executed cell summarized outputs
    }
    """
    notebook_path = Path(notebook_path).expanduser().resolve()

    with open(notebook_path, "r") as f:
        nb = nbformat.read(f, as_version=4)

    exec_nb = nbformat.v4.new_notebook(
        metadata=nb.metadata,
        cells=nb.cells[: upto_index + 1],
    )

    client = NotebookClient(exec_nb, timeout=timeout, kernel_name="python3", allow_errors=True)

    try:
        client.execute()
    except Exception:
        # allow_errors=Trueなので outputs に error が入る想定
        pass

    # Summarize outputs per cell (stdout/stderr/result/display/error)
    cell_outputs = []
    failing_info = None

    for i, cell in enumerate(exec_nb.cells):
        if cell.get("cell_type") != "code":
            cell_outputs.append({"index": i, "cell_type": cell.get("cell_type"), "outputs": []})
            continue

        outs_summary = []
        for out in cell.get("outputs", []) or []:
            ot = out.get("output_type")
            if ot == "stream":
                outs_summary.append({
                    "type": "stream",
                    "name": out.get("name"),
                    "text": out.get("text", "")[:4000],  # avoid huge spam
                })
            elif ot in ("execute_result", "display_data"):
                # show plain text if exists
                data = out.get("data", {}) or {}
                txt = data.get("text/plain", "")
                outs_summary.append({
                    "type": ot,
                    "text": (txt or "")[:2000],
                })
            elif ot == "error":
                tb = "\n".join(out.get("traceback", []) or [])
                outs_summary.append({
                    "type": "error",
                    "ename": out.get("ename"),
                    "evalue": out.get("evalue"),
                    "traceback": tb[:8000],
                })
                if failing_info is None:
                    failing_info = {
                        "failing_cell": i,
                        "exception_type": out.get("ename"),
                        "exception_message": out.get("evalue"),
                        "traceback": tb,
                    }
            else:
                outs_summary.append({"type": ot})

        cell_outputs.append({"index": i, "cell_type": "code", "outputs": outs_summary})

    if failing_info:
        return False, {"cell_outputs": cell_outputs, **failing_info}

    return True, {
        "failing_cell": None,
        "exception_type": None,
        "exception_message": None,
        "traceback": None,
        "cell_outputs": cell_outputs,
    }

def render_cell_outputs(exec_info, max_chars_per_output=800):
    """
    exec_info is the dict returned by execute_notebook_prefix_with_outputs (success info).
    Prints per-cell outputs in Markdown for quick debugging.
    """
    blocks = []
    for c in exec_info.get("cell_outputs", []):
        idx = c["index"]
        blocks.append(f"### Cell index {idx}")
        outs = c.get("outputs", [])

        if not outs:
            blocks.append("- (no outputs)")
            continue

        for j, o in enumerate(outs, start=1):
            t = o.get("type")
            if t == "stream":
                name = o.get("name")
                txt = (o.get("text") or "")[:max_chars_per_output]
                blocks.append(f"- **stream ({name})**\n\n```text\n{txt}\n```")
            elif t in ("execute_result", "display_data"):
                txt = (o.get("text") or "")[:max_chars_per_output]
                blocks.append(f"- **{t}**\n\n```text\n{txt}\n```")
            elif t == "error":
                en = o.get("ename")
                ev = o.get("evalue")
                tb = (o.get("traceback") or "")[:max_chars_per_output]
                blocks.append(f"- ❌ **error {en}: {ev}**\n\n```text\n{tb}\n```")
            else:
                blocks.append(f"- **{t}**")

    display(Markdown("\n\n".join(blocks)))
    
import json

def apply_claude_repair_patch_for_cell(
    target_notebook_path: str | Path,
    failing_cell_index: int,
    repair_user_prompt: str,
):
    """
    Ask Claude to produce a JSON patch, then apply it to the target notebook.
    The patch must only update failing_cell_index.
    """
    resp = claude_client.messages.create(
        model=CLAUDE_PATCH_MODEL,
        system=CLAUDE_PATCH_SYSTEM,
        messages=[{"role": "user", "content": repair_user_prompt}],
        max_tokens=int(CLAUDE_PATCH_MAX_TOKENS),
    )

    parts = []
    for block in resp.content:
        if getattr(block, "type", None) == "text":
            parts.append(block.text)
        else:
            parts.append(str(block))
    raw = "\n".join(parts).strip()

    json_str = _extract_first_json_object(raw)
    patch = json.loads(json_str)

    apply_patch_to_notebook(Path(target_notebook_path), patch, enforce_header_prefix=True)
    return patch

import re
from pathlib import Path

def build_and_test_loop_cellwise(
    *,
    target_notebook_path: str | Path,
    cell00_content: str,
    structure_lines: list[str],
    max_fix_attempts_per_step: int = 3,
    context_upto: int = 14,
    show_outputs: str = "on_fail",   # "always" | "on_fail" | "never"
):
    """
    Cell-wise loop:
      - Patch the next planned cell
      - Execute prefix (Cell 00 .. current cell)
      - If error: repair failing cell and re-run prefix
      - If clean: proceed to next cell
    """

    # Parse structure goals: "Cell 01: Title"
    cell_goals = {}
    for line in structure_lines:
        m = re.match(r"Cell (\d+): (.+)", line.strip())
        if m:
            cell_goals[int(m.group(1))] = m.group(2).strip()

    planned_cells = sorted([i for i in cell_goals.keys() if i > 0])
    target_notebook_path = Path(target_notebook_path).expanduser().resolve()

    def _read_cell_source(idx: int) -> str:
        with open(target_notebook_path, "r") as f:
            nb = nbformat.read(f, as_version=4)
        return nb.cells[idx].source if idx < len(nb.cells) else ""

    print("🧪 Cell-wise build-and-test loop starting...")
    print("Target notebook:", target_notebook_path)
    print("Planned cells:", planned_cells)

    for current_cell in planned_cells:
        print("\n" + "=" * 70)
        print(f"➡️ Step: Generate Cell {current_cell:02} — {cell_goals.get(current_cell,'')}")
        print("=" * 70)

        # 1) Patch current cell
        patch = call_claude_patch_for_cells(
            notebook_path=target_notebook_path,
            target_indices=[current_cell],
            context_upto=context_upto,
        )
        apply_patch_to_notebook(target_notebook_path, patch, enforce_header_prefix=True)
        print(f"✅ Patched Cell {current_cell:02} (initial).")

        # 2) Execute prefix up to current_cell
        for attempt in range(1, max_fix_attempts_per_step + 1):
            print(f"[Run] Execute prefix 0..{current_cell} (attempt {attempt}/{max_fix_attempts_per_step})")

            ok, info = execute_notebook_prefix_with_outputs(
                target_notebook_path,
                upto_index=current_cell,
            )

            # Decide whether to show outputs
            if info is not None:
                if show_outputs == "always" or (show_outputs == "on_fail" and not ok):
                    render_cell_outputs(info)

            if ok:
                print(f"✅ Prefix execution OK up to Cell {current_cell:02}. Proceeding.")
                break

            # Build minimal error dict
            err = {
                "failing_cell": (info or {}).get("failing_cell"),
                "exception_type": (info or {}).get("exception_type"),
                "exception_message": (info or {}).get("exception_message"),
                "traceback": (info or {}).get("traceback"),
            }

            failing = err["failing_cell"]
            etype = err["exception_type"]
            emsg = err["exception_message"]
            print(f"❌ Failed at prefix cell index={failing}: {etype}: {emsg}")

            if attempt >= max_fix_attempts_per_step:
                print("❌ Max fix attempts reached for this step. Stopping.")
                return False

            # 3) Create repair prompt for the failing cell
            failing_goal = cell_goals.get(
                failing,
                f"Fix Cell {int(failing or 0):02} so that the notebook prefix executes without errors.",
            )
            failing_src = _read_cell_source(int(failing or 0))

            repair_prompt = generate_repair_prompt_with_openai(
                cell00_content=cell00_content,
                cell_goal=failing_goal,
                current_source=failing_src,
                error_info=err,
            )
            print("🛠 Repair prompt (head):", repair_prompt[:180], "...")

            # 4) Apply repair patch to failing cell (may be different from current_cell)
            apply_claude_repair_patch_for_cell(
                target_notebook_path=target_notebook_path,
                failing_cell_index=int(failing or 0),
                repair_user_prompt=repair_prompt,
            )
            print(f"✅ Applied repair patch to failing Cell {int(failing or 0):02}.")

    print("\n🎉 Done. All planned cells patched and validated cell-by-cell.")
    return True


print("✅ Phase 2 build-and-test loop functions defined")

✅ Phase 2 build-and-test loop functions defined


In [24]:
# ============================================================
# Cell 09 — Phase 2 Smoke Test Runner (with Outputs)
# ============================================================
# Overview:
# Smoke test for Phase 2. Shows per-cell outputs so you can verify
# whether the notebook is actually healthy before proceeding.
#
# Inputs / Outputs:
# Inputs:
# - LAST_SAVED_NOTEBOOK_PATH
# - execute_notebook_prefix_with_outputs
# - render_cell_outputs
# - build_and_test_loop_cellwise
#
# Outputs:
# - Printed structure
# - Per-cell outputs for prefix executions
# - Final result boolean
#
# Notes:
# - Use RUN_MODE="dry" first to only execute and inspect outputs.
# - Then RUN_MODE="phase2" to actually patch+repair cellwise.
#

# --- Safety checks ---
required_globals = [
    "LAST_SAVED_NOTEBOOK_PATH",
    "execute_notebook_prefix_with_outputs",
    "render_cell_outputs",
    "build_and_test_loop_cellwise",
]
for g in required_globals:
    if g not in globals():
        raise RuntimeError(f"Missing required global: {g}")

from pathlib import Path
import nbformat

target_nb_path = Path(LAST_SAVED_NOTEBOOK_PATH).expanduser().resolve()

# Load Cell 00 + Structure info
with open(target_nb_path, "r") as f:
    nb = nbformat.read(f, as_version=4)

cell00_content = nb.cells[0].source

# Extract Structure lines from Cell 00
structure_lines = [
    line.strip("# ").strip()
    for line in cell00_content.splitlines()
    if line.strip().startswith("# Cell ")
]

print("🔍 Detected structure:")
for l in structure_lines:
    print("  -", l)

# ----------------------------
# Choose mode
# ----------------------------
RUN_MODE = "phase2"     # "dry" or "phase2"
UPTO_CELL = 100        # for dry mode: execute prefix up to this cell index (inclusive)
SHOW_OUTPUTS = True  # print outputs every run

# ----------------------------
# Dry run: execute prefix and show outputs
# ----------------------------
if RUN_MODE == "dry":
    print("\n🧪 DRY RUN: execute prefix and show outputs")
    ok, info = execute_notebook_prefix_with_outputs(target_nb_path, upto_index=UPTO_CELL)

    if SHOW_OUTPUTS:
        render_cell_outputs(info)

    if ok:
        print(f"✅ Dry run OK (0..{UPTO_CELL})")
    else:
        print(f"❌ Dry run failed at cell index={info['failing_cell']}")
        print(f"   {info['exception_type']}: {info['exception_message']}")

    print("Result:", ok)

# ----------------------------
# Phase2 run: patch+repair cellwise
# ----------------------------
elif RUN_MODE == "phase2":
    print("\n🚀 PHASE2 RUN: patch+repair cellwise")
    ok = build_and_test_loop_cellwise(
        target_notebook_path=target_nb_path,
        cell00_content=cell00_content,
        structure_lines=structure_lines,
        max_fix_attempts_per_step=3,
        context_upto=14,
    )
    print("Result:", ok)

else:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")


🔍 Detected structure:
  - Cell 01: Imports, environment setup, constants, logging, run_id
  - Cell 02: Notion API wrappers and schema helpers
  - Cell 03: Time window computation (last 7 days, JST)
  - Cell 04: Fetch weekly events from Events data source
  - Cell 05: Normalize records and validate fields
  - Cell 06: Deduplication and noise filtering
  - Cell 07: Theme clustering and optional LLM labeling
  - Cell 08: Theme scoring and top N selection
  - Cell 09: Render weekly digest markdown
  - Cell 10: Notion writeback upsert to Weekly Digests
  - Cell 11: Export local artifacts and final summary

🚀 PHASE2 RUN: patch+repair cellwise
🧪 Cell-wise build-and-test loop starting...
Target notebook: /Users/yuetoya/projects/researchOS100-private/notebooks/041_weekly_events_digest.ipynb
Planned cells: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

➡️ Step: Generate Cell 01 — Imports, environment setup, constants, logging, run_id
✅ Patched Cell 01 (initial).
[Run] Execute prefix 0..1 (attempt 1/3)
✅ P